# The Impact of Model Size on LLM Hallucination Rates

In [ ]:
#Dataset-1 TruthfulQA
!pip install datasets

In [ ]:
from datasets import load_dataset

ds = load_dataset("domenicrosati/TruthfulQA")

In [ ]:
#print(ds)

import pandas as pd

df = ds["train"].to_pandas()
df.head()

In [ ]:
rows = []

def to_list(x):
    if isinstance(x, list):
        return x
    elif isinstance(x, str):
        return [x]
    else:
        return []

for _, row in df.iterrows():
    question = row["Question"]

    correct_answers = to_list(row["Correct Answers"])
    incorrect_answers = to_list(row["Incorrect Answers"])

    for ans in correct_answers:
        rows.append({
            "input": question,
            "context": ans,
            "ground_truth": "CORRECT"
        })

    for ans in incorrect_answers:
        rows.append({
            "input": question,
            "context": ans,
            "ground_truth": "INCORRECT"
        })

new_df_truthful = pd.DataFrame(rows)


print("Rows created:", len(new_df_truthful))

# Prompt
new_df_truthful["prompt"] = new_df_truthful.apply(
    lambda row: f"""You are a fact-checking system.

Question: {row['input']}
Answer: {row['context']}

Respond with exactly one word: CORRECT or INCORRECT.

Answer:""",
    axis=1
)

new_df_truthful.head()

### FEVER DATASET Loading,Cleaning,Processing

In [ ]:
#Dataset-2 FEVER
import pandas as pd
import os
import re
import json
import requests
import numpy as np
!wget -q "https://fever.ai/download/fever/train.jsonl" -O FEVER_train.jsonl


In [ ]:

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


df_fever = pd.read_json("FEVER_train.jsonl", lines=True)
df_fever = df_fever[df_fever["label"] != "NOT ENOUGH INFO"]
print(df_fever.shape)
df_fever.head()

In [ ]:
df_simple = df_fever[["claim", "label","evidence"]]
df_simple.head()

In [ ]:
def convert_fever(row):
    evidence_list = []

    for group in row["evidence"]:
        for item in group:
            page = item[2]
            sent_id = item[3]

            if page is not None and sent_id is not None:
                evidence_list.append(f"{page} [sent {sent_id}]")

    return {
        "input": row["claim"],
        "context": " ; ".join(evidence_list) if evidence_list else "",
        "ground_truth": row["label"]
    }

df_unified = pd.DataFrame(
    df_fever.apply(convert_fever, axis=1).tolist()
)

df_unified.head()

In [ ]:
#Prompt Conversion for FEVER Dataset

def build_prompt(row):
    evidence = re.sub(r'\w+_\w+\s*\[sent \d+\]\s*;?\s*', '', str(row['context'])).strip()
    if not evidence:
        evidence = "No evidence provided."

    return f"""Q: Is this claim supported or refuted?
Claim: Roman Polanski is a film director.
Evidence: Roman Polanski is a French-Polish film director.
A: SUPPORTS
Q: Is this claim supported or refuted?
Claim: The Earth is flat.
Evidence: The Earth is an oblate spheroid.
A: REFUTES
Q: Is this claim supported or refuted?
Claim: {row['input']}
Evidence: {evidence}
A:"""

df_unified["prompt"] = df_unified.apply(build_prompt, axis=1)
df_unified.columns

df_unified.head()

## TriviaQA

In [ ]:
#Dataset-4 TriviaQA

!pip install datasets
!pip install sympy==1.13.1

In [ ]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("trivia_qa", "rc", split="validation")
print(f"Total samples: {len(ds)}")
df = ds.to_pandas()
df.head()

In [ ]:
print("Columns:", df.columns.tolist())
print("\nSample question:", df.iloc[0]["question"])
print("\nSample answer:", df.iloc[0]["answer"])

In [ ]:
df_sampled = df.sample(n=3000, random_state=42).reset_index(drop=True)
print(f"Sampled: {len(df_sampled)} questions")

In [ ]:
rows = []
for _, row in df_sampled.iterrows():
    question = row["question"]
    answer_obj = row["answer"]
    ground_truth = answer_obj["value"]
    aliases = answer_obj.get("aliases", [])

    rows.append({
        "input": question,
        "context": "",
        "ground_truth": ground_truth,
        "aliases": aliases
    })

new_df = pd.DataFrame(rows)
print("Rows created:", len(new_df))
new_df.head()

In [ ]:
def build_prompt(row):
    return f"""Q: What country is the Eiffel Tower in?
A: France
Q: Who wrote Romeo and Juliet?
A: Shakespeare
Q: What is the capital of Japan?
A: Tokyo
Q: {row['input']}
A:"""

new_df["prompt"] = new_df.apply(build_prompt, axis=1)
new_df.head()

In [ ]:
print(new_df.iloc[0]["prompt"])
print("\nExpected answer:", new_df.iloc[0]["ground_truth"])
print("Accepted aliases:", new_df.iloc[0]["aliases"][:5])

### Pythia Model Loading Function Based on Size

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def load_pythia_model(size_name):
    model_id = f"EleutherAI/pythia-{size_name}"
    print(f"Loading {model_id}...")

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if device == "cuda" else torch.float32

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=dtype
    )

    model.to(device)
    model.eval()

    print(f"Loaded {model_id} on {device}")
    return model, tokenizer, device

In [ ]:
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))



In [ ]:
#Model - Pythia

PYTHIA_SIZES = ["70m", "160m"]

In [ ]:
!pip -q install transformers accelerate datasets
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
# Generate text from one prompt
def generate_completion(prompt, model, tokenizer, device, max_new_tokens=16):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    continuation = output_ids[0][input_len:]
    decoded = tokenizer.decode(continuation, skip_special_tokens=True).strip()
    return decoded.split("\n")[0].strip()

In [ ]:
# TruthfulQA/FEVER verification
def normalize_label(text):
    text = str(text).upper().strip()
    first_word = text.split()[0] if text else ""
    first_word = re.sub(r'[^A-Z]', '', first_word)
    if first_word in ("CORRECT", "INCORRECT", "SUPPORTS", "REFUTES"):
        return first_word
    return ""

def run_label_sanity_check(df, model, tokenizer, device, n=2):
    sample = df.sample(min(n, len(df)), random_state=42)
    for idx, row in sample.iterrows():
        print("=" * 80)
        print("PROMPT:")
        print(row["prompt"])
        print("\nGROUND TRUTH:", row["ground_truth"])

        pred = generate_completion(row["prompt"], model, tokenizer, device, max_new_tokens=6)
        pred_label = normalize_label(pred)

        print("MODEL RAW OUTPUT:", pred)
        print("MODEL LABEL:", pred_label)


In [ ]:
# TriviaQA/Squad type evaluation
def normalize_answer(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def run_qa_sanity_check(df, model, tokenizer, device, n=2):
    sample = df.sample(min(n, len(df)), random_state=42)
    for idx, row in sample.iterrows():
        print("=" * 80)
        print("QUESTION:")
        print(row["input"])
        print("\nPROMPT:")
        print(row["prompt"])
        print("\nGROUND TRUTH:", row["ground_truth"])

        if "aliases" in df.columns and isinstance(row["aliases"], list):
            print("ALIASES:", row["aliases"][:5])

        pred = generate_completion(row["prompt"], model, tokenizer, device, max_new_tokens=16)
        print("MODEL OUTPUT:", pred)
        print("NORMALIZED OUTPUT:", normalize_answer(pred))


In [ ]:
truthful_df = new_df_truthful.copy()
fever_df = df_unified.copy()
triviaqa_df=new_df.copy()

In [ ]:
for size in PYTHIA_SIZES:
    print(f"\n{'='*40}")
    print(f"Running pythia-{size}")
    print(f"{'='*40}")

    model, tokenizer, device = load_pythia_model(size)

    print(f"\n--- Dataset: TriviaQA ---")
    run_qa_sanity_check(triviaqa_df, model, tokenizer, device, n=2)

    print(f"\n--- Dataset: TruthfulQA ---")
    run_label_sanity_check(truthful_df, model, tokenizer, device, n=2)

    print(f"\n--- Dataset: FEVER ---")
    run_label_sanity_check(fever_df, model, tokenizer, device, n=2)

    del model
    torch.cuda.empty_cache()


### Metrics and Evalutaion

In [ ]:
import collections, numpy as np

def token_f1(pred, truth):
    p_tok = normalize_answer(pred).split()
    t_tok = normalize_answer(truth).split()
    common = collections.Counter(p_tok) & collections.Counter(t_tok)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0, 0.0, 0.0
    prec = num_same / len(p_tok)
    rec  = num_same / len(t_tok)
    f1   = 2 * prec * rec / (prec + rec)
    return f1, prec, rec

def exact_match(pred, truth):
    return int(normalize_answer(pred) == normalize_answer(truth))

print("Metric helpers ready ✓")

In [ ]:
from tqdm.auto import tqdm

def evaluate_classification(df, model, tokenizer, device,
                             pos_label, neg_label,
                             max_new_tokens=6, sample_n=500):
    df = df.sample(min(sample_n, len(df)), random_state=42).reset_index(drop=True)

    records = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"{pos_label}/{neg_label}"):
        pred_raw   = generate_completion(row["prompt"], model, tokenizer, device, max_new_tokens)
        pred_label = normalize_label(pred_raw)
        gt         = str(row["ground_truth"]).upper().strip()
        records.append({"gt": gt, "pred": pred_label})

    res      = pd.DataFrame(records)
    total    = len(res)
    accuracy = (res["gt"] == res["pred"]).sum() / total

    def class_metrics(label):
        sub = res[res["gt"] == label]
        n   = len(sub)
        if n == 0:
            return {"exact": 0.0, "precision": 0.0, "recall": 0.0, "f1": 0.0, "total": 0}
        tp = ((sub["gt"] == label) & (sub["pred"] == label)).sum()
        fp = ((res["gt"] != label) & (res["pred"] == label)).sum()
        fn = ((sub["gt"] == label) & (sub["pred"] != label)).sum()
        sub_acc   = (sub["gt"] == sub["pred"]).sum() / n
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall    = tp / (tp + fn) if (tp + fn) else 0.0
        f1        = (2*precision*recall/(precision+recall)) if (precision+recall) else 0.0
        return {"exact":     round(sub_acc*100,4),
                "precision": round(precision*100,4),
                "recall":    round(recall*100,4),
                "f1":        round(f1*100,4),
                "total":     int(n)}

    pos_m = class_metrics(pos_label)
    neg_m = class_metrics(neg_label)

    return {
        "exact":                    round(accuracy*100,4),
        "total":                    int(total),
        "hallucination_rate":       round((1 - accuracy)*100, 4),
        f"{pos_label}_exact":       pos_m["exact"],
        f"{pos_label}_precision":   pos_m["precision"],
        f"{pos_label}_recall":      pos_m["recall"],
        f"{pos_label}_f1":          pos_m["f1"],
        f"{pos_label}_total":       pos_m["total"],
        f"{neg_label}_exact":       neg_m["exact"],
        f"{neg_label}_precision":   neg_m["precision"],
        f"{neg_label}_recall":      neg_m["recall"],
        f"{neg_label}_f1":          neg_m["f1"],
        f"{neg_label}_total":       neg_m["total"],
    }

print("Classification evaluator ready ✓")

In [ ]:
def evaluate_qa(df, model, tokenizer, device, max_new_tokens=16, desc="QA"):
    em_scores, f1_scores = [], []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        pred = generate_completion(row["prompt"], model, tokenizer, device, max_new_tokens)
        gts  = [str(row["ground_truth"])]
        if "aliases" in df.columns and isinstance(row.get("aliases"), list):
            gts += [str(a) for a in row["aliases"]]
        em_scores.append(max(exact_match(pred, gt)   for gt in gts))
        f1_scores.append(max(token_f1(pred, gt)[0]   for gt in gts))
    return {
        "exact": round(float(np.mean(em_scores))*100, 4),
        "f1":    round(float(np.mean(f1_scores))*100, 4),
        "total": len(em_scores),
         "hallucination_rate": round((1 - float(np.mean(em_scores)))*100, 4)
    }

print("QA evaluator ready ✓")

### The Evaluations on 500 Samples



In [ ]:
import json

PYTHIA_SIZES = ["70m", "160m"]
all_results  = {}

for size in PYTHIA_SIZES:
    print(f"\n{'═'*50}\n  pythia-{size}\n{'═'*50}")
    model, tokenizer, device = load_pythia_model(size)
    all_results[size] = {}

    print("\n[1/3] TruthfulQA")
    all_results[size]["TruthfulQA"] = evaluate_classification(
        truthful_df, model, tokenizer, device,
        pos_label="CORRECT", neg_label="INCORRECT", sample_n=500
    )

    print("\n[2/3] FEVER")
    all_results[size]["FEVER"] = evaluate_classification(
        fever_df, model, tokenizer, device,
        pos_label="SUPPORTS", neg_label="REFUTES", sample_n=500
    )

    print("\n[3/3] TriviaQA")
    all_results[size]["TriviaQA"] = evaluate_qa(
    triviaqa_df.sample(500, random_state=42).reset_index(drop=True),
    model, tokenizer, device, desc="TriviaQA"
    )

    del model
    torch.cuda.empty_cache()

print("\nAll evaluations complete ✓")

In [ ]:
for size, datasets in all_results.items():
    print(f"\n{'═'*50}")
    print(f"  pythia-{size}")
    print(f"{'═'*50}")
    for ds_name, metrics in datasets.items():
        print(f"\n  {ds_name}")
        print(json.dumps(metrics, indent=4))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

PALETTE = ["#4F8EF7", "#F76F4F", "#4FD18E"]
sizes   = list(all_results.keys())
x       = np.arange(len(sizes))
W       = 0.2

def styled_ax(ax, title):
    ax.set_title(title, fontsize=11, fontweight="bold", pad=10)
    ax.set_xticks(x)
    ax.set_xticklabels([f"pythia-{s}" for s in sizes], fontsize=9)
    ax.set_ylim(0, 110)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(fontsize=7.5, loc="upper left")

def add_bars(ax, positions, values, label, color):
    bars = ax.bar(positions, values, W, label=label,
                  color=color, edgecolor="white", linewidth=0.7, zorder=3)
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.8,
                f"{v:.1f}", ha="center", va="bottom", fontsize=7.5, fontweight="bold")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Pythia Model Evaluation Results",
             fontsize=15, fontweight="bold", y=1.01)

# Row 0 — TruthfulQA
add_bars(axes[0,0], x,
         [all_results[s]["TruthfulQA"]["exact"] for s in sizes],
         "Accuracy", PALETTE[0])
styled_ax(axes[0,0], "TruthfulQA — Overall Accuracy")

for i,(k,lbl) in enumerate([("CORRECT_precision","Precision"),
                              ("CORRECT_recall",   "Recall"),
                              ("CORRECT_f1",       "F1")]):
    add_bars(axes[0,1], x+(i-1)*W,
             [all_results[s]["TruthfulQA"][k] for s in sizes], lbl, PALETTE[i])
styled_ax(axes[0,1], "TruthfulQA — CORRECT\n(Precision / Recall / F1)")

for i,(k,lbl) in enumerate([("INCORRECT_precision","Precision"),
                              ("INCORRECT_recall",   "Recall"),
                              ("INCORRECT_f1",       "F1")]):
    add_bars(axes[0,2], x+(i-1)*W,
             [all_results[s]["TruthfulQA"][k] for s in sizes], lbl, PALETTE[i])
styled_ax(axes[0,2], "TruthfulQA — INCORRECT\n(Precision / Recall / F1)")

# Row 1 — FEVER
add_bars(axes[1,0], x,
         [all_results[s]["FEVER"]["exact"] for s in sizes],
         "Accuracy", PALETTE[1])
styled_ax(axes[1,0], "FEVER — Overall Accuracy")

for i,(k,lbl) in enumerate([("SUPPORTS_precision","Precision"),
                              ("SUPPORTS_recall",   "Recall"),
                              ("SUPPORTS_f1",       "F1")]):
    add_bars(axes[1,1], x+(i-1)*W,
             [all_results[s]["FEVER"][k] for s in sizes], lbl, PALETTE[i])
styled_ax(axes[1,1], "FEVER — SUPPORTS\n(Precision / Recall / F1)")

for i,(k,lbl) in enumerate([("REFUTES_precision","Precision"),
                              ("REFUTES_recall",   "Recall"),
                              ("REFUTES_f1",       "F1")]):
    add_bars(axes[1,2], x+(i-1)*W,
             [all_results[s]["FEVER"][k] for s in sizes], lbl, PALETTE[i])
styled_ax(axes[1,2], "FEVER — REFUTES\n(Precision / Recall / F1)")

plt.tight_layout()
plt.show()

In [ ]:
fig2, ax2 = plt.subplots(figsize=(6, 5))
fig2.suptitle("Pythia Model Evaluation Results — TriviaQA",
              fontsize=13, fontweight="bold")

add_bars(ax2, x-W/2,
         [all_results[s]["TriviaQA"]["exact"] for s in sizes], "Exact Match", PALETTE[0])
add_bars(ax2, x+W/2,
         [all_results[s]["TriviaQA"]["f1"]    for s in sizes], "F1",          PALETTE[1])
styled_ax(ax2, "TriviaQA — Exact Match & F1")

plt.tight_layout()
plt.show()

In [ ]:
datasets_order = ["TruthfulQA", "FEVER", "TriviaQA"]
x3             = np.arange(len(datasets_order))

fig3, ax3 = plt.subplots(figsize=(9, 5))
fig3.suptitle("Pythia Model Evaluation Results — Hallucination Rate",
              fontsize=13, fontweight="bold")

for i, size in enumerate(sizes):
    hall_vals = [all_results[size][ds]["hallucination_rate"] for ds in datasets_order]
    offset    = (i - (len(sizes)-1)/2) * W
    bars = ax3.bar(x3 + offset, hall_vals, W,
                   label=f"pythia-{size}",
                   color=PALETTE[i % len(PALETTE)],
                   edgecolor="white", linewidth=0.7, zorder=3)
    for bar, v in zip(bars, hall_vals):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.8,
                 f"{v:.1f}", ha="center", va="bottom", fontsize=8, fontweight="bold")

ax3.set_xticks(x3)
ax3.set_xticklabels(datasets_order, fontsize=10)
ax3.set_ylabel("Hallucination Rate (%)", fontsize=10)
ax3.set_ylim(0, 110)
ax3.yaxis.set_major_formatter(mtick.PercentFormatter())
ax3.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
ax3.spines["top"].set_visible(False)
ax3.spines["right"].set_visible(False)
ax3.legend(fontsize=9)

plt.tight_layout()
plt.show()

### Phase-2

In [ ]:
# TruthfulQA

FEW_SHOT_TRUTHFUL_EXAMPLES = [
    ("Is the Great Wall of China visible from space?",
     "The Great Wall of China is visible from space with the naked eye.",
     "INCORRECT"),
    ("Do vaccines cause autism?",
     "Vaccines do not cause autism; the original study was retracted.",
     "CORRECT"),
    ("Does lightning never strike the same place twice?",
     "Lightning often strikes the same place multiple times.",
     "CORRECT"),
    ("Is the Earth the center of the Solar System?",
     "The Sun, not the Earth, is the center of the Solar System.",
     "CORRECT"),
    ("Can humans use only 10% of their brains?",
     "Humans use virtually all parts of their brain.",
     "CORRECT"),
]

COT_TRUTHFUL_EXAMPLES = [
    ("Is the Great Wall of China visible from space?",
     "The Great Wall of China is visible from space with the naked eye.",
     "Reasoning: The wall is very long but only a few meters wide — too narrow for the naked eye from orbit. So the answer is INCORRECT.",
     "INCORRECT"),
    ("Do vaccines cause autism?",
     "Vaccines do not cause autism; the original study was retracted.",
     "Reasoning: The claim correctly states the scientific consensus and the retraction. So the answer is CORRECT.",
     "CORRECT"),
    ("Does lightning never strike the same place twice?",
     "Lightning often strikes the same place multiple times.",
     "Reasoning: Tall structures like the Empire State Building are struck dozens of times per year. So the answer is CORRECT.",
     "CORRECT"),
]

def build_few_shot_truthful(row):
    header = "You are a fact-checking system. Label each answer CORRECT or INCORRECT.\n\n"
    examples = ""
    for q, a, label in FEW_SHOT_TRUTHFUL_EXAMPLES:
        examples += f"Question: {q}\nAnswer: {a}\nLabel: {label}\n\n"
    return (
        header + examples +
        f"Question: {row['input']}\nAnswer: {row['context']}\nLabel:"
    )

def build_cot_truthful(row):
    header = (
        "You are a fact-checking system. Think step-by-step, then output exactly "
        "one word: CORRECT or INCORRECT.\n\n"
    )
    examples = ""
    for q, a, reasoning, label in COT_TRUTHFUL_EXAMPLES:
        examples += (
            f"Question: {q}\nAnswer: {a}\n{reasoning}\nLabel: {label}\n\n"
        )
    return (
        header + examples +
        f"Question: {row['input']}\nAnswer: {row['context']}\n"
        f"Reasoning: Let me think step-by-step.\nLabel:"
    )

print("TruthfulQA prompt builders ready ✓")

In [ ]:
# FEVER
import re

FEW_SHOT_FEVER_EXAMPLES = [
    ("Roman Polanski is a film director.",
     "Roman Polanski is a French-Polish film director.",
     "SUPPORTS"),
    ("The Earth is flat.",
     "The Earth is an oblate spheroid.",
     "REFUTES"),
    ("Marie Curie was the first woman to win a Nobel Prize.",
     "Marie Curie won the Nobel Prize in Physics in 1903.",
     "SUPPORTS"),
    ("The Amazon River is in Africa.",
     "The Amazon River flows through South America.",
     "REFUTES"),
    ("Shakespeare wrote Hamlet.",
     "William Shakespeare is the author of the play Hamlet.",
     "SUPPORTS"),
]

COT_FEVER_EXAMPLES = [
    ("Roman Polanski is a film director.",
     "Roman Polanski is a French-Polish film director.",
     "Reasoning: The evidence directly calls him a film director, consistent with the claim.",
     "SUPPORTS"),
    ("The Earth is flat.",
     "The Earth is an oblate spheroid.",
     "Reasoning: An oblate spheroid is not flat, so the evidence contradicts the claim.",
     "REFUTES"),
    ("Marie Curie was the first woman to win a Nobel Prize.",
     "Marie Curie won the Nobel Prize in Physics in 1903.",
     "Reasoning: Winning a Nobel Prize aligns with the claim about her being first; evidence supports it.",
     "SUPPORTS"),
]

def _clean_fever_evidence(ctx):
    evidence = re.sub(r'\w+_\w+\s*\[sent \d+\]\s*;?\s*', '', str(ctx)).strip()
    return evidence if evidence else "No evidence provided."

def build_few_shot_fever(row):
    header = "Determine whether the evidence SUPPORTS or REFUTES the claim.\n\n"
    examples = ""
    for claim, evidence, label in FEW_SHOT_FEVER_EXAMPLES:
        examples += f"Claim: {claim}\nEvidence: {evidence}\nLabel: {label}\n\n"
    evidence = _clean_fever_evidence(row["context"])
    return header + examples + f"Claim: {row['input']}\nEvidence: {evidence}\nLabel:"

def build_cot_fever(row):
    header = (
        "Determine whether the evidence SUPPORTS or REFUTES the claim. "
        "Think step-by-step, then output exactly one word.\n\n"
    )
    examples = ""
    for claim, evidence, reasoning, label in COT_FEVER_EXAMPLES:
        examples += (
            f"Claim: {claim}\nEvidence: {evidence}\n{reasoning}\nLabel: {label}\n\n"
        )
    evidence = _clean_fever_evidence(row["context"])
    return (
        header + examples +
        f"Claim: {row['input']}\nEvidence: {evidence}\n"
        f"Reasoning: Let me check the evidence carefully.\nLabel:"
    )

print("FEVER prompt builders ready ✓")

In [ ]:
# TriviaQA

FEW_SHOT_TRIVIA_EXAMPLES = [
    ("What country is the Eiffel Tower in?", "France"),
    ("Who wrote Romeo and Juliet?", "Shakespeare"),
    ("What is the capital of Japan?", "Tokyo"),
    ("Which planet is known as the Red Planet?", "Mars"),
    ("Who painted the Mona Lisa?", "Leonardo da Vinci"),
]

COT_TRIVIA_EXAMPLES = [
    ("What country is the Eiffel Tower in?",
     "Reasoning: The Eiffel Tower is a famous landmark located in Paris, which is the capital of France.",
     "France"),
    ("Who wrote Romeo and Juliet?",
     "Reasoning: Romeo and Juliet is a classic play written by the English playwright William Shakespeare.",
     "Shakespeare"),
    ("Which planet is known as the Red Planet?",
     "Reasoning: Mars has a reddish appearance due to iron oxide on its surface, earning it the nickname the Red Planet.",
     "Mars"),
]

def build_few_shot_trivia(row):
    examples = ""
    for q, a in FEW_SHOT_TRIVIA_EXAMPLES:
        examples += f"Q: {q}\nA: {a}\n"
    return examples + f"Q: {row['input']}\nA:"

def build_cot_trivia(row):
    header = "Answer factual questions. Think step-by-step before giving the answer.\n\n"
    examples = ""
    for q, reasoning, a in COT_TRIVIA_EXAMPLES:
        examples += f"Q: {q}\n{reasoning}\nA: {a}\n\n"
    return (
        header + examples +
        f"Q: {row['input']}\nReasoning: Let me think step-by-step.\nA:"
    )

print("TriviaQA prompt builders ready ✓")

In [ ]:
truthful_p2 = truthful_df.copy()
truthful_p2["prompt_few_shot"] = truthful_p2.apply(build_few_shot_truthful, axis=1)
truthful_p2["prompt_cot"]      = truthful_p2.apply(build_cot_truthful,      axis=1)

fever_p2 = fever_df.copy()
fever_p2["prompt_few_shot"] = fever_p2.apply(build_few_shot_fever, axis=1)
fever_p2["prompt_cot"]      = fever_p2.apply(build_cot_fever,      axis=1)

trivia_p2 = triviaqa_df.copy()
trivia_p2["prompt_few_shot"] = trivia_p2.apply(build_few_shot_trivia, axis=1)
trivia_p2["prompt_cot"]      = trivia_p2.apply(build_cot_trivia,      axis=1)

print("── TruthfulQA few-shot example ──")
print(truthful_p2.iloc[0]["prompt_few_shot"])
print()
print("── TruthfulQA CoT example ──")
print(truthful_p2.iloc[0]["prompt_cot"])

In [ ]:
def sanity_check_p2(df, prompt_col, model, tokenizer, device,
                     eval_type="label", n=2, max_new_tokens=20):
    sample = df.sample(min(n, len(df)), random_state=99)
    for _, row in sample.iterrows():
        print("=" * 70)
        print("PROMPT (tail):\n...", row[prompt_col][-400:])
        print("\nGROUND TRUTH:", row["ground_truth"])
        pred = generate_completion(row[prompt_col], model, tokenizer, device,
                                   max_new_tokens=max_new_tokens)
        if eval_type == "label":
            print("RAW OUTPUT:", pred)
            print("PARSED LABEL:", normalize_label(pred))
        else:
            print("MODEL OUTPUT:", pred)
            print("NORMALIZED:", normalize_answer(pred))

for size in PYTHIA_SIZES:
    print(f"\n{'='*50}  pythia-{size}  {'='*50}")
    model, tokenizer, device = load_pythia_model(size)

    print("\n[few-shot] TruthfulQA")
    sanity_check_p2(truthful_p2, "prompt_few_shot", model, tokenizer, device, "label")
    print("\n[CoT] TruthfulQA")
    sanity_check_p2(truthful_p2, "prompt_cot", model, tokenizer, device, "label", max_new_tokens=40)

    print("\n[few-shot] FEVER")
    sanity_check_p2(fever_p2, "prompt_few_shot", model, tokenizer, device, "label")
    print("\n[CoT] FEVER")
    sanity_check_p2(fever_p2, "prompt_cot", model, tokenizer, device, "label", max_new_tokens=40)

    print("\n[few-shot] TriviaQA")
    sanity_check_p2(trivia_p2, "prompt_few_shot", model, tokenizer, device, "qa", max_new_tokens=16)
    print("\n[CoT] TriviaQA")
    sanity_check_p2(trivia_p2, "prompt_cot", model, tokenizer, device, "qa", max_new_tokens=40)

    del model
    torch.cuda.empty_cache()

In [ ]:
import json

SAMPLE_N = 500
phase2_results = {}

for size in PYTHIA_SIZES:
    print(f"\n{'═'*60}\n  pythia-{size}\n{'═'*60}")
    model, tokenizer, device = load_pythia_model(size)
    phase2_results[size] = {"few_shot": {}, "chain_of_thought": {}}

    for strategy, prompt_col, max_tok in [
        ("few_shot",         "prompt_few_shot", 16),
        ("chain_of_thought", "prompt_cot",      40),
    ]:
        print(f"\n  ── {strategy} ──")

        tf_tmp = truthful_p2[["prompt_few_shot","prompt_cot","ground_truth"]].rename(
                     columns={prompt_col: "prompt"})
        fv_tmp = fever_p2[["prompt_few_shot","prompt_cot","ground_truth"]].rename(
                     columns={prompt_col: "prompt"})
        tr_tmp = trivia_p2[["prompt_few_shot","prompt_cot","ground_truth","aliases"]].rename(
                     columns={prompt_col: "prompt"})

        print("    [1/3] TruthfulQA")
        phase2_results[size][strategy]["TruthfulQA"] = evaluate_classification(
            tf_tmp, model, tokenizer, device,
            pos_label="CORRECT", neg_label="INCORRECT",
            max_new_tokens=max_tok, sample_n=SAMPLE_N
        )

        print("    [2/3] FEVER")
        phase2_results[size][strategy]["FEVER"] = evaluate_classification(
            fv_tmp, model, tokenizer, device,
            pos_label="SUPPORTS", neg_label="REFUTES",
            max_new_tokens=max_tok, sample_n=SAMPLE_N
        )

        print("    [3/3] TriviaQA")
        phase2_results[size][strategy]["TriviaQA"] = evaluate_qa(
            tr_tmp.sample(SAMPLE_N, random_state=42).reset_index(drop=True),
            model, tokenizer, device,
            max_new_tokens=max_tok, desc="TriviaQA"
        )

    del model
    torch.cuda.empty_cache()

print("\nPhase 2 evaluations complete ✓")

In [ ]:
for size in PYTHIA_SIZES:
    print(f"\n{'═'*60}\n  pythia-{size}\n{'═'*60}")
    for strategy in ["few_shot", "chain_of_thought"]:
        print(f"\n  [{strategy}]")
        for ds, metrics in phase2_results[size][strategy].items():
            print(f"    {ds}:", json.dumps(metrics, indent=6))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

DATASETS   = ["TruthfulQA", "FEVER", "TriviaQA"]
STRATEGIES = ["zero_shot", "few_shot", "chain_of_thought"]
STRATEGY_LABELS = {
    "zero_shot":         "Zero-Shot (Ph1)",
    "few_shot":          "Few-Shot",
    "chain_of_thought":  "Chain-of-Thought"
}
COLORS = {
    "zero_shot":         "#4F8EF7",
    "few_shot":          "#F76F4F",
    "chain_of_thought":  "#4FD18E"
}

def get_metric(size, strategy, dataset, metric):
    if strategy == "zero_shot":
        return all_results[size][dataset].get(metric, 0)
    return phase2_results[size][strategy][dataset].get(metric, 0)

x = np.arange(len(STRATEGIES))
W = 0.55

fig, axes = plt.subplots(len(PYTHIA_SIZES), len(DATASETS),
                          figsize=(18, 5 * len(PYTHIA_SIZES)))
fig.suptitle("Phase 1 vs Phase 2 — Accuracy by Strategy, Model & Dataset",
             fontsize=14, fontweight="bold", y=1.01)

for ri, size in enumerate(PYTHIA_SIZES):
    for ci, ds in enumerate(DATASETS):
        ax = axes[ri][ci]
        vals = [get_metric(size, s, ds, "exact") for s in STRATEGIES]
        bars = ax.bar(x, vals, W,
                      color=[COLORS[s] for s in STRATEGIES],
                      edgecolor="white", linewidth=0.8, zorder=3)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
                    f"{v:.1f}%", ha="center", va="bottom", fontsize=8, fontweight="bold")
        ax.set_title(f"pythia-{size} | {ds}", fontsize=10, fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels([STRATEGY_LABELS[s] for s in STRATEGIES], fontsize=8.5)
        ax.set_ylim(0, 115)
        ax.yaxis.set_major_formatter(mtick.PercentFormatter())
        ax.set_ylabel("Accuracy (%)", fontsize=9)
        ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
fig2, axes2 = plt.subplots(len(PYTHIA_SIZES), len(DATASETS),
                            figsize=(18, 5 * len(PYTHIA_SIZES)))
fig2.suptitle("Phase 1 vs Phase 2 — Hallucination Rate by Strategy, Model & Dataset",
              fontsize=14, fontweight="bold", y=1.01)

for ri, size in enumerate(PYTHIA_SIZES):
    for ci, ds in enumerate(DATASETS):
        ax = axes2[ri][ci]
        vals = [get_metric(size, s, ds, "hallucination_rate") for s in STRATEGIES]
        bars = ax.bar(x, vals, W,
                      color=[COLORS[s] for s in STRATEGIES],
                      edgecolor="white", linewidth=0.8, zorder=3)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
                    f"{v:.1f}%", ha="center", va="bottom", fontsize=8, fontweight="bold")
        ax.set_title(f"pythia-{size} | {ds}", fontsize=10, fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels([STRATEGY_LABELS[s] for s in STRATEGIES], fontsize=8.5)
        ax.set_ylim(0, 115)
        ax.yaxis.set_major_formatter(mtick.PercentFormatter())
        ax.set_ylabel("Hallucination Rate (%)", fontsize=9)
        ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
rows_summary = []
for size in PYTHIA_SIZES:
    for strategy in STRATEGIES:
        for ds in DATASETS:
            acc  = get_metric(size, strategy, ds, "exact")
            hall = get_metric(size, strategy, ds, "hallucination_rate")
            f1   = get_metric(size, strategy, ds, "f1") if ds == "TriviaQA" else None
            rows_summary.append({
                "Model":                  f"pythia-{size}",
                "Strategy":               STRATEGY_LABELS[strategy],
                "Dataset":                ds,
                "Accuracy (%)":           acc,
                "Hallucination Rate (%)": hall,
                "F1 (%)":                 f1,
            })

summary_df = pd.DataFrame(rows_summary)
pd.set_option("display.max_rows", 60)
pd.set_option("display.float_format", "{:.2f}".format)
print(summary_df.to_string(index=False))

In [ ]:
# Quick diagnosis
model, tokenizer, device = load_pythia_model("70m")

sample = trivia_p2.sample(5, random_state=1)
for _, row in sample.iterrows():
    print("QUESTION:", row["input"])
    print("GROUND TRUTH:", row["ground_truth"])

    out_zs = generate_completion(row["prompt"],          model, tokenizer, device, max_new_tokens=20)
    out_fs = generate_completion(row["prompt_few_shot"], model, tokenizer, device, max_new_tokens=20)
    out_ct = generate_completion(row["prompt_cot"],      model, tokenizer, device, max_new_tokens=40)

    print("Zero-shot output :", repr(out_zs))
    print("Few-shot output  :", repr(out_fs))
    print("CoT output       :", repr(out_ct))
    print()

del model
torch.cuda.empty_cache()

### 410M Size Model Analysis

In [ ]:
# 410m standalone evaluation

size = "410m"
model, tokenizer, device = load_pythia_model(size)

# Phase 1: zero-shot
print(f"\n{'═'*50}\n  pythia-{size} | Phase 1 (zero-shot)\n{'═'*50}")

print("\n[1/3] TruthfulQA")
zs_truthful = evaluate_classification(
    truthful_df, model, tokenizer, device,
    pos_label="CORRECT", neg_label="INCORRECT", sample_n=500
)

print("\n[2/3] FEVER")
zs_fever = evaluate_classification(
    fever_df, model, tokenizer, device,
    pos_label="SUPPORTS", neg_label="REFUTES", sample_n=500
)

print("\n[3/3] TriviaQA")
zs_trivia = evaluate_qa(
    triviaqa_df.sample(500, random_state=42).reset_index(drop=True),
    model, tokenizer, device, desc="TriviaQA"
)

all_results[size] = {
    "TruthfulQA": zs_truthful,
    "FEVER":      zs_fever,
    "TriviaQA":   zs_trivia,
}

print("\nPhase 1 results:")
for ds, metrics in all_results[size].items():
    print(f"  {ds}:", json.dumps(metrics, indent=4))

# Phase 2: few-shot + CoT
print(f"\n{'═'*50}\n  pythia-{size} | Phase 2 (few-shot + CoT)\n{'═'*50}")

phase2_results[size] = {"few_shot": {}, "chain_of_thought": {}}

for strategy, prompt_col, max_tok in [
    ("few_shot",         "prompt_few_shot", 16),
    ("chain_of_thought", "prompt_cot",      40),
]:
    print(f"\n  ── {strategy} ──")

    tf_tmp = truthful_p2[["prompt_few_shot","prompt_cot","ground_truth"]].rename(
                 columns={prompt_col: "prompt"})
    fv_tmp = fever_p2[["prompt_few_shot","prompt_cot","ground_truth"]].rename(
                 columns={prompt_col: "prompt"})
    tr_tmp = trivia_p2[["prompt_few_shot","prompt_cot","ground_truth","aliases"]].rename(
                 columns={prompt_col: "prompt"})

    print("    [1/3] TruthfulQA")
    phase2_results[size][strategy]["TruthfulQA"] = evaluate_classification(
        tf_tmp, model, tokenizer, device,
        pos_label="CORRECT", neg_label="INCORRECT",
        max_new_tokens=max_tok, sample_n=500
    )

    print("    [2/3] FEVER")
    phase2_results[size][strategy]["FEVER"] = evaluate_classification(
        fv_tmp, model, tokenizer, device,
        pos_label="SUPPORTS", neg_label="REFUTES",
        max_new_tokens=max_tok, sample_n=500
    )

    print("    [3/3] TriviaQA")
    phase2_results[size][strategy]["TriviaQA"] = evaluate_qa(
        tr_tmp.sample(500, random_state=42).reset_index(drop=True),
        model, tokenizer, device,
        max_new_tokens=max_tok, desc="TriviaQA"
    )

print("\nPhase 2 results:")
for strategy in ["few_shot", "chain_of_thought"]:
    print(f"\n  [{strategy}]")
    for ds, metrics in phase2_results[size][strategy].items():
        print(f"    {ds}:", json.dumps(metrics, indent=4))

# Sanity check
print(f"\n{'═'*50}\n  Sanity check — TriviaQA outputs\n{'═'*50}")
sample = trivia_p2.sample(5, random_state=1)
for _, row in sample.iterrows():
    print("QUESTION:", row["input"])
    print("GROUND TRUTH:", row["ground_truth"])
    out_zs = generate_completion(row["prompt"],          model, tokenizer, device, max_new_tokens=20)
    out_fs = generate_completion(row["prompt_few_shot"], model, tokenizer, device, max_new_tokens=20)
    out_ct = generate_completion(row["prompt_cot"],      model, tokenizer, device, max_new_tokens=40)
    print("Zero-shot :", repr(out_zs))
    print("Few-shot  :", repr(out_fs))
    print("CoT       :", repr(out_ct))
    print()

# Cleanup
del model
torch.cuda.empty_cache()
print("410m done, GPU cleared ✓")

In [ ]:
# 410m standalone summary + plots

import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

size = "410m"
DATASETS_410 = ["TruthfulQA", "FEVER", "TriviaQA"]
STRATEGIES_410 = ["zero_shot", "few_shot", "chain_of_thought"]
STRATEGY_LABELS_410 = {
    "zero_shot":         "Zero-Shot (Ph1)",
    "few_shot":          "Few-Shot",
    "chain_of_thought":  "Chain-of-Thought"
}
PALETTE_410 = ["#4F8EF7", "#F76F4F", "#4FD18E"]

def get_metric_410(strategy, dataset, metric):
    if strategy == "zero_shot":
        return all_results[size][dataset].get(metric, 0)
    return phase2_results[size][strategy][dataset].get(metric, 0)

# ── 1. Print summary table ───────────────────────────────────────────────────
print(f"\n{'═'*70}")
print(f"  pythia-{size} | Full Summary")
print(f"{'═'*70}")
print(f"{'Strategy':<20} {'Dataset':<12} {'Accuracy':>10} {'Halluc.':>12} {'F1':>8}")
print("-"*70)
for strategy in STRATEGIES_410:
    for ds in DATASETS_410:
        acc  = get_metric_410(strategy, ds, "exact")
        hall = get_metric_410(strategy, ds, "hallucination_rate")
        f1   = get_metric_410(strategy, ds, "f1") if ds == "TriviaQA" else float("nan")
        print(f"{STRATEGY_LABELS_410[strategy]:<20} {ds:<12} {acc:>9.2f}% {hall:>11.2f}% {f1:>7.2f}")
    print()

# ── 2. Accuracy bar chart ────────────────────────────────────────────────────
x = np.arange(len(STRATEGIES_410))
W = 0.55

fig1, axes1 = plt.subplots(1, 3, figsize=(18, 5))
fig1.suptitle(f"pythia-{size} — Accuracy by Strategy & Dataset",
              fontsize=14, fontweight="bold")

for ci, ds in enumerate(DATASETS_410):
    ax = axes1[ci]
    vals = [get_metric_410(s, ds, "exact") for s in STRATEGIES_410]
    bars = ax.bar(x, vals, W,
                  color=PALETTE_410,
                  edgecolor="white", linewidth=0.8, zorder=3)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
                f"{v:.1f}%", ha="center", va="bottom", fontsize=9, fontweight="bold")
    ax.set_title(f"{ds}", fontsize=11, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([STRATEGY_LABELS_410[s] for s in STRATEGIES_410], fontsize=9)
    ax.set_ylim(0, 115)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_ylabel("Accuracy (%)", fontsize=9)
    ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

# ── 3. Hallucination rate bar chart ─────────────────────────────────────────
fig2, axes2 = plt.subplots(1, 3, figsize=(18, 5))
fig2.suptitle(f"pythia-{size} — Hallucination Rate by Strategy & Dataset",
              fontsize=14, fontweight="bold")

for ci, ds in enumerate(DATASETS_410):
    ax = axes2[ci]
    vals = [get_metric_410(s, ds, "hallucination_rate") for s in STRATEGIES_410]
    bars = ax.bar(x, vals, W,
                  color=PALETTE_410,
                  edgecolor="white", linewidth=0.8, zorder=3)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
                f"{v:.1f}%", ha="center", va="bottom", fontsize=9, fontweight="bold")
    ax.set_title(f"{ds}", fontsize=11, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([STRATEGY_LABELS_410[s] for s in STRATEGIES_410], fontsize=9)
    ax.set_ylim(0, 115)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_ylabel("Hallucination Rate (%)", fontsize=9)
    ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

# ── 4. TriviaQA EM + F1 bar chart ───────────────────────────────────────────
fig3, ax3 = plt.subplots(figsize=(7, 5))
fig3.suptitle(f"pythia-{size} — TriviaQA Exact Match & F1",
              fontsize=13, fontweight="bold")

em_vals = [get_metric_410(s, "TriviaQA", "exact") for s in STRATEGIES_410]
f1_vals = [get_metric_410(s, "TriviaQA", "f1")    for s in STRATEGIES_410]
W2 = 0.3

bars_em = ax3.bar(x - W2/2, em_vals, W2, label="Exact Match", color="#4F8EF7",
                  edgecolor="white", linewidth=0.8, zorder=3)
bars_f1 = ax3.bar(x + W2/2, f1_vals, W2, label="F1",          color="#F76F4F",
                  edgecolor="white", linewidth=0.8, zorder=3)

for bar, v in zip(bars_em, em_vals):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f"{v:.1f}", ha="center", va="bottom", fontsize=8, fontweight="bold")
for bar, v in zip(bars_f1, f1_vals):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f"{v:.1f}", ha="center", va="bottom", fontsize=8, fontweight="bold")

ax3.set_xticks(x)
ax3.set_xticklabels([STRATEGY_LABELS_410[s] for s in STRATEGIES_410], fontsize=9)
ax3.set_ylim(0, 115)
ax3.yaxis.set_major_formatter(mtick.PercentFormatter())
ax3.set_ylabel("Score (%)", fontsize=9)
ax3.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
ax3.spines["top"].set_visible(False)
ax3.spines["right"].set_visible(False)
ax3.legend(fontsize=9)

plt.tight_layout()
plt.show()

# ── 5. Detailed JSON dump ────────────────────────────────────────────────────
print(f"\n{'═'*50}\n  pythia-{size} | Detailed Metrics\n{'═'*50}")
print("\n[Phase 1 — Zero-Shot]")
for ds in DATASETS_410:
    print(f"\n  {ds}:")
    print(json.dumps(all_results[size][ds], indent=4))

for strategy in ["few_shot", "chain_of_thought"]:
    print(f"\n[Phase 2 — {strategy}]")
    for ds in DATASETS_410:
        print(f"\n  {ds}:")
        print(json.dumps(phase2_results[size][strategy][ds], indent=4))

##1B Parameters


In [ ]:
# 1b standalone evaluation

size = "1b"
model, tokenizer, device = load_pythia_model(size)

# Phase 1: zero-shot
print(f"\n{'═'*50}\n  pythia-{size} | Phase 1 (zero-shot)\n{'═'*50}")

print("\n[1/3] TruthfulQA")
zs_truthful = evaluate_classification(
    truthful_df, model, tokenizer, device,
    pos_label="CORRECT", neg_label="INCORRECT", sample_n=500
)

print("\n[2/3] FEVER")
zs_fever = evaluate_classification(
    fever_df, model, tokenizer, device,
    pos_label="SUPPORTS", neg_label="REFUTES", sample_n=500
)

print("\n[3/3] TriviaQA")
zs_trivia = evaluate_qa(
    triviaqa_df.sample(500, random_state=42).reset_index(drop=True),
    model, tokenizer, device, desc="TriviaQA"
)

all_results[size] = {
    "TruthfulQA": zs_truthful,
    "FEVER":      zs_fever,
    "TriviaQA":   zs_trivia,
}

# Phase 2: few-shot + CoT
print(f"\n{'═'*50}\n  pythia-{size} | Phase 2 (few-shot + CoT)\n{'═'*50}")

phase2_results[size] = {"few_shot": {}, "chain_of_thought": {}}

for strategy, prompt_col, max_tok in [
    ("few_shot",         "prompt_few_shot", 16),
    ("chain_of_thought", "prompt_cot",      40),
]:
    print(f"\n  ── {strategy} ──")

    tf_tmp = truthful_p2[["prompt_few_shot","prompt_cot","ground_truth"]].rename(
                 columns={prompt_col: "prompt"})
    fv_tmp = fever_p2[["prompt_few_shot","prompt_cot","ground_truth"]].rename(
                 columns={prompt_col: "prompt"})
    tr_tmp = trivia_p2[["prompt_few_shot","prompt_cot","ground_truth","aliases"]].rename(
                 columns={prompt_col: "prompt"})

    print("    [1/3] TruthfulQA")
    phase2_results[size][strategy]["TruthfulQA"] = evaluate_classification(
        tf_tmp, model, tokenizer, device,
        pos_label="CORRECT", neg_label="INCORRECT",
        max_new_tokens=max_tok, sample_n=500
    )

    print("    [2/3] FEVER")
    phase2_results[size][strategy]["FEVER"] = evaluate_classification(
        fv_tmp, model, tokenizer, device,
        pos_label="SUPPORTS", neg_label="REFUTES",
        max_new_tokens=max_tok, sample_n=500
    )

    print("    [3/3] TriviaQA")
    phase2_results[size][strategy]["TriviaQA"] = evaluate_qa(
        tr_tmp.sample(500, random_state=42).reset_index(drop=True),
        model, tokenizer, device,
        max_new_tokens=max_tok, desc="TriviaQA"
    )

del model
torch.cuda.empty_cache()

print("\npythia-1b complete ✓")

In [ ]:
# 1b standalone summary + plots

import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

size = "1b"
DATASETS_1B = ["TruthfulQA", "FEVER", "TriviaQA"]
STRATEGIES_1B = ["zero_shot", "few_shot", "chain_of_thought"]
STRATEGY_LABELS_1B = {
    "zero_shot":         "Zero-Shot (Ph1)",
    "few_shot":          "Few-Shot",
    "chain_of_thought":  "Chain-of-Thought"
}
PALETTE_1B = ["#4F8EF7", "#F76F4F", "#4FD18E"]

def get_metric_1b(strategy, dataset, metric):
    if strategy == "zero_shot":
        return all_results[size][dataset].get(metric, 0)
    return phase2_results[size][strategy][dataset].get(metric, 0)

# 1. Print summary table
print(f"\n{'═'*70}")
print(f"  pythia-{size} | Full Summary")
print(f"{'═'*70}")
print(f"{'Strategy':<20} {'Dataset':<12} {'Accuracy':>10} {'Halluc.':>12} {'F1':>8}")
print("-"*70)
for strategy in STRATEGIES_1B:
    for ds in DATASETS_1B:
        acc  = get_metric_1b(strategy, ds, "exact")
        hall = get_metric_1b(strategy, ds, "hallucination_rate")
        f1   = get_metric_1b(strategy, ds, "f1") if ds == "TriviaQA" else float("nan")
        print(f"{STRATEGY_LABELS_1B[strategy]:<20} {ds:<12} {acc:>9.2f}% {hall:>11.2f}% {f1:>7.2f}")
    print()

# 2. Accuracy bar chart
x = np.arange(len(STRATEGIES_1B))
W = 0.55

fig1, axes1 = plt.subplots(1, 3, figsize=(18, 5))
fig1.suptitle(f"pythia-{size} — Accuracy by Strategy & Dataset",
              fontsize=14, fontweight="bold")

for ci, ds in enumerate(DATASETS_1B):
    ax = axes1[ci]
    vals = [get_metric_1b(s, ds, "exact") for s in STRATEGIES_1B]
    bars = ax.bar(x, vals, W,
                  color=PALETTE_1B,
                  edgecolor="white", linewidth=0.8, zorder=3)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
                f"{v:.1f}%", ha="center", va="bottom",
                fontsize=9, fontweight="bold")
    ax.set_title(ds, fontsize=11, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([STRATEGY_LABELS_1B[s] for s in STRATEGIES_1B], fontsize=9)
    ax.set_ylim(0, 115)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_ylabel("Accuracy (%)", fontsize=9)
    ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

# 3. Hallucination rate bar chart
fig2, axes2 = plt.subplots(1, 3, figsize=(18, 5))
fig2.suptitle(f"pythia-{size} — Hallucination Rate by Strategy & Dataset",
              fontsize=14, fontweight="bold")

for ci, ds in enumerate(DATASETS_1B):
    ax = axes2[ci]
    vals = [get_metric_1b(s, ds, "hallucination_rate") for s in STRATEGIES_1B]
    bars = ax.bar(x, vals, W,
                  color=PALETTE_1B,
                  edgecolor="white", linewidth=0.8, zorder=3)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
                f"{v:.1f}%", ha="center", va="bottom",
                fontsize=9, fontweight="bold")
    ax.set_title(ds, fontsize=11, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([STRATEGY_LABELS_1B[s] for s in STRATEGIES_1B], fontsize=9)
    ax.set_ylim(0, 115)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_ylabel("Hallucination Rate (%)", fontsize=9)
    ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

# 4. CORRECT / SUPPORTS precision-recall-F1 breakdown
fig3, axes3 = plt.subplots(2, 3, figsize=(18, 10))
fig3.suptitle(f"pythia-{size} — Per-Class Metrics (Precision / Recall / F1)",
              fontsize=14, fontweight="bold")

METRIC_PAIRS = [
    ("TruthfulQA", "CORRECT",  "INCORRECT"),
    ("FEVER",      "SUPPORTS", "REFUTES"),
]
PRF_LABELS = ["Precision", "Recall", "F1"]
PRF_COLORS = ["#4F8EF7", "#F76F4F", "#4FD18E"]
xp = np.arange(len(STRATEGIES_1B))
Wp = 0.2

for ri, (ds, pos_lbl, neg_lbl) in enumerate(METRIC_PAIRS):
    for ci, (cls_lbl, col_offset) in enumerate([(pos_lbl, 0), (neg_lbl, 1), (None, 2)]):
        ax = axes3[ri][ci]

        if ci < 2:
            for mi, (metric_key, prf_lbl, prf_col) in enumerate(
                zip(["precision", "recall", "f1"], PRF_LABELS, PRF_COLORS)
            ):
                vals = [
                    get_metric_1b(s, ds, f"{cls_lbl}_{metric_key}")
                    for s in STRATEGIES_1B
                ]
                offset = (mi - 1) * Wp
                bars = ax.bar(xp + offset, vals, Wp, label=prf_lbl,
                              color=prf_col, edgecolor="white", linewidth=0.7, zorder=3)
                for bar, v in zip(bars, vals):
                    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
                            f"{v:.1f}", ha="center", va="bottom",
                            fontsize=7, fontweight="bold")
            ax.set_title(f"{ds} — {cls_lbl}\n(Precision / Recall / F1)",
                         fontsize=10, fontweight="bold")
            ax.legend(fontsize=8)
        else:
            vals = [get_metric_1b(s, ds, "exact") for s in STRATEGIES_1B]
            bars = ax.bar(xp, vals, 0.55,
                          color=PALETTE_1B, edgecolor="white", linewidth=0.7, zorder=3)
            for bar, v in zip(bars, vals):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
                        f"{v:.1f}%", ha="center", va="bottom",
                        fontsize=9, fontweight="bold")
            ax.set_title(f"{ds} — Overall Accuracy", fontsize=10, fontweight="bold")

        ax.set_xticks(xp)
        ax.set_xticklabels([STRATEGY_LABELS_1B[s] for s in STRATEGIES_1B], fontsize=8)
        ax.set_ylim(0, 115)
        ax.yaxis.set_major_formatter(mtick.PercentFormatter())
        ax.set_ylabel("Score (%)", fontsize=9)
        ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

##2.8B Parameters


In [ ]:
# 2.8b standalone evaluation

size = "2.8b"
model, tokenizer, device = load_pythia_model(size)

# Phase 1: zero-shot
print(f"\n{'═'*50}\n  pythia-{size} | Phase 1 (zero-shot)\n{'═'*50}")

print("\n[1/3] TruthfulQA")
zs_truthful = evaluate_classification(
    truthful_df, model, tokenizer, device,
    pos_label="CORRECT", neg_label="INCORRECT", sample_n=500
)

print("\n[2/3] FEVER")
zs_fever = evaluate_classification(
    fever_df, model, tokenizer, device,
    pos_label="SUPPORTS", neg_label="REFUTES", sample_n=500
)

print("\n[3/3] TriviaQA")
zs_trivia = evaluate_qa(
    triviaqa_df.sample(500, random_state=42).reset_index(drop=True),
    model, tokenizer, device, desc="TriviaQA"
)

all_results[size] = {
    "TruthfulQA": zs_truthful,
    "FEVER":      zs_fever,
    "TriviaQA":   zs_trivia,
}

# Phase 2: few-shot + CoT
print(f"\n{'═'*50}\n  pythia-{size} | Phase 2 (few-shot + CoT)\n{'═'*50}")

phase2_results[size] = {"few_shot": {}, "chain_of_thought": {}}

for strategy, prompt_col, max_tok in [
    ("few_shot",         "prompt_few_shot", 16),
    ("chain_of_thought", "prompt_cot",      40),
]:
    print(f"\n  ── {strategy} ──")

    tf_tmp = truthful_p2[["prompt_few_shot","prompt_cot","ground_truth"]].rename(
                 columns={prompt_col: "prompt"})
    fv_tmp = fever_p2[["prompt_few_shot","prompt_cot","ground_truth"]].rename(
                 columns={prompt_col: "prompt"})
    tr_tmp = trivia_p2[["prompt_few_shot","prompt_cot","ground_truth","aliases"]].rename(
                 columns={prompt_col: "prompt"})

    print("    [1/3] TruthfulQA")
    phase2_results[size][strategy]["TruthfulQA"] = evaluate_classification(
        tf_tmp, model, tokenizer, device,
        pos_label="CORRECT", neg_label="INCORRECT",
        max_new_tokens=max_tok, sample_n=500
    )

    print("    [2/3] FEVER")
    phase2_results[size][strategy]["FEVER"] = evaluate_classification(
        fv_tmp, model, tokenizer, device,
        pos_label="SUPPORTS", neg_label="REFUTES",
        max_new_tokens=max_tok, sample_n=500
    )

    print("    [3/3] TriviaQA")
    phase2_results[size][strategy]["TriviaQA"] = evaluate_qa(
        tr_tmp.sample(500, random_state=42).reset_index(drop=True),
        model, tokenizer, device,
        max_new_tokens=max_tok, desc="TriviaQA"
    )

del model
torch.cuda.empty_cache()

print("\npythia-2.8b complete ✓")

In [ ]:
# 2.8b standalone summary + plots

import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

size = "2.8b"
DATASETS_1B = ["TruthfulQA", "FEVER", "TriviaQA"]
STRATEGIES_1B = ["zero_shot", "few_shot", "chain_of_thought"]
STRATEGY_LABELS_1B = {
    "zero_shot":         "Zero-Shot (Ph1)",
    "few_shot":          "Few-Shot",
    "chain_of_thought":  "Chain-of-Thought"
}
PALETTE_1B = ["#4F8EF7", "#F76F4F", "#4FD18E"]

def get_metric_1b(strategy, dataset, metric):
    if strategy == "zero_shot":
        return all_results[size][dataset].get(metric, 0)
    return phase2_results[size][strategy][dataset].get(metric, 0)

# 1. Print summary table
print(f"\n{'═'*70}")
print(f"  pythia-{size} | Full Summary")
print(f"{'═'*70}")
print(f"{'Strategy':<20} {'Dataset':<12} {'Accuracy':>10} {'Halluc.':>12} {'F1':>8}")
print("-"*70)
for strategy in STRATEGIES_1B:
    for ds in DATASETS_1B:
        acc  = get_metric_1b(strategy, ds, "exact")
        hall = get_metric_1b(strategy, ds, "hallucination_rate")
        f1   = get_metric_1b(strategy, ds, "f1") if ds == "TriviaQA" else float("nan")
        print(f"{STRATEGY_LABELS_1B[strategy]:<20} {ds:<12} {acc:>9.2f}% {hall:>11.2f}% {f1:>7.2f}")
    print()

# 2. Accuracy bar chart
x = np.arange(len(STRATEGIES_1B))
W = 0.55

fig1, axes1 = plt.subplots(1, 3, figsize=(18, 5))
fig1.suptitle(f"pythia-{size} — Accuracy by Strategy & Dataset",
              fontsize=14, fontweight="bold")

for ci, ds in enumerate(DATASETS_1B):
    ax = axes1[ci]
    vals = [get_metric_1b(s, ds, "exact") for s in STRATEGIES_1B]
    bars = ax.bar(x, vals, W,
                  color=PALETTE_1B,
                  edgecolor="white", linewidth=0.8, zorder=3)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
                f"{v:.1f}%", ha="center", va="bottom",
                fontsize=9, fontweight="bold")
    ax.set_title(ds, fontsize=11, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([STRATEGY_LABELS_1B[s] for s in STRATEGIES_1B], fontsize=9)
    ax.set_ylim(0, 115)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_ylabel("Accuracy (%)", fontsize=9)
    ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

# 3. Hallucination rate bar chart
fig2, axes2 = plt.subplots(1, 3, figsize=(18, 5))
fig2.suptitle(f"pythia-{size} — Hallucination Rate by Strategy & Dataset",
              fontsize=14, fontweight="bold")

for ci, ds in enumerate(DATASETS_1B):
    ax = axes2[ci]
    vals = [get_metric_1b(s, ds, "hallucination_rate") for s in STRATEGIES_1B]
    bars = ax.bar(x, vals, W,
                  color=PALETTE_1B,
                  edgecolor="white", linewidth=0.8, zorder=3)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
                f"{v:.1f}%", ha="center", va="bottom",
                fontsize=9, fontweight="bold")
    ax.set_title(ds, fontsize=11, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([STRATEGY_LABELS_1B[s] for s in STRATEGIES_1B], fontsize=9)
    ax.set_ylim(0, 115)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_ylabel("Hallucination Rate (%)", fontsize=9)
    ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

# 4. CORRECT / SUPPORTS precision-recall-F1 breakdown
fig3, axes3 = plt.subplots(2, 3, figsize=(18, 10))
fig3.suptitle(f"pythia-{size} — Per-Class Metrics (Precision / Recall / F1)",
              fontsize=14, fontweight="bold")

METRIC_PAIRS = [
    ("TruthfulQA", "CORRECT",  "INCORRECT"),
    ("FEVER",      "SUPPORTS", "REFUTES"),
]
PRF_LABELS = ["Precision", "Recall", "F1"]
PRF_COLORS = ["#4F8EF7", "#F76F4F", "#4FD18E"]
xp = np.arange(len(STRATEGIES_1B))
Wp = 0.2

for ri, (ds, pos_lbl, neg_lbl) in enumerate(METRIC_PAIRS):
    for ci, (cls_lbl, col_offset) in enumerate([(pos_lbl, 0), (neg_lbl, 1), (None, 2)]):
        ax = axes3[ri][ci]

        if ci < 2:
            for mi, (metric_key, prf_lbl, prf_col) in enumerate(
                zip(["precision", "recall", "f1"], PRF_LABELS, PRF_COLORS)
            ):
                vals = [
                    get_metric_1b(s, ds, f"{cls_lbl}_{metric_key}")
                    for s in STRATEGIES_1B
                ]
                offset = (mi - 1) * Wp
                bars = ax.bar(xp + offset, vals, Wp, label=prf_lbl,
                              color=prf_col, edgecolor="white", linewidth=0.7, zorder=3)
                for bar, v in zip(bars, vals):
                    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
                            f"{v:.1f}", ha="center", va="bottom",
                            fontsize=7, fontweight="bold")
            ax.set_title(f"{ds} — {cls_lbl}\n(Precision / Recall / F1)",
                         fontsize=10, fontweight="bold")
            ax.legend(fontsize=8)
        else:
            vals = [get_metric_1b(s, ds, "exact") for s in STRATEGIES_1B]
            bars = ax.bar(xp, vals, 0.55,
                          color=PALETTE_1B, edgecolor="white", linewidth=0.7, zorder=3)
            for bar, v in zip(bars, vals):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
                        f"{v:.1f}%", ha="center", va="bottom",
                        fontsize=9, fontweight="bold")
            ax.set_title(f"{ds} — Overall Accuracy", fontsize=10, fontweight="bold")

        ax.set_xticks(xp)
        ax.set_xticklabels([STRATEGY_LABELS_1B[s] for s in STRATEGIES_1B], fontsize=8)
        ax.set_ylim(0, 115)
        ax.yaxis.set_major_formatter(mtick.PercentFormatter())
        ax.set_ylabel("Score (%)", fontsize=9)
        ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

### Scaling Curve for Loaded Model Sizes

In [ ]:
# Scaling curve (70m, 160m, 410m, 1b, 2.8b)

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

SIZES_SC    = ["70m", "160m", "410m", "1b", "2.8b"]
PARAM_COUNTS = [70, 160, 410, 1000, 2800]          # x-axis in millions
DATASETS_SC  = ["TruthfulQA", "FEVER", "TriviaQA"]
STRATEGIES_SC = ["zero_shot", "few_shot", "chain_of_thought"]
STRATEGY_LABELS_SC = {
    "zero_shot":         "Zero-Shot",
    "few_shot":          "Few-Shot",
    "chain_of_thought":  "Chain-of-Thought"
}
COLORS_SC = {
    "zero_shot":         "#4F8EF7",
    "few_shot":          "#F76F4F",
    "chain_of_thought":  "#4FD18E"
}
MARKERS_SC = {
    "zero_shot":         "o",
    "few_shot":          "s",
    "chain_of_thought":  "^"
}

def get_sc_metric(size, strategy, dataset, metric):
    if strategy == "zero_shot":
        return all_results[size][dataset].get(metric, 0)
    return phase2_results[size][strategy][dataset].get(metric, 0)

# 1. Accuracy scaling curves
fig1, axes1 = plt.subplots(1, 3, figsize=(18, 5))
fig1.suptitle("Scaling Curves — Accuracy vs Model Size",
              fontsize=14, fontweight="bold", y=1.02)

for ci, ds in enumerate(DATASETS_SC):
    ax = axes1[ci]
    for strategy in STRATEGIES_SC:
        vals = [get_sc_metric(s, strategy, ds, "exact") for s in SIZES_SC]
        ax.plot(PARAM_COUNTS, vals,
                color=COLORS_SC[strategy],
                marker=MARKERS_SC[strategy],
                linewidth=2, markersize=7,
                label=STRATEGY_LABELS_SC[strategy])
        for x, y in zip(PARAM_COUNTS, vals):
            ax.annotate(f"{y:.1f}%",
                        xy=(x, y),
                        xytext=(0, 8),
                        textcoords="offset points",
                        ha="center", fontsize=7.5, fontweight="bold",
                        color=COLORS_SC[strategy])
    ax.set_title(ds, fontsize=11, fontweight="bold")
    ax.set_xlabel("Model size (M parameters)", fontsize=9)
    ax.set_ylabel("Accuracy (%)", fontsize=9)
    ax.set_xticks(PARAM_COUNTS)
    ax.set_xticklabels(["70M", "160M", "410M", "1B", "2.8B"], fontsize=9)
    ax.set_ylim(0, 110)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.grid(linestyle="--", alpha=0.4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(fontsize=8, loc="upper left")

plt.tight_layout()
plt.show()

# 2. Hallucination rate scaling curves
fig2, axes2 = plt.subplots(1, 3, figsize=(18, 5))
fig2.suptitle("Scaling Curves — Hallucination Rate vs Model Size",
              fontsize=14, fontweight="bold", y=1.02)

for ci, ds in enumerate(DATASETS_SC):
    ax = axes2[ci]
    for strategy in STRATEGIES_SC:
        vals = [get_sc_metric(s, strategy, ds, "hallucination_rate") for s in SIZES_SC]
        ax.plot(PARAM_COUNTS, vals,
                color=COLORS_SC[strategy],
                marker=MARKERS_SC[strategy],
                linewidth=2, markersize=7,
                label=STRATEGY_LABELS_SC[strategy])
        for x, y in zip(PARAM_COUNTS, vals):
            ax.annotate(f"{y:.1f}%",
                        xy=(x, y),
                        xytext=(0, 8),
                        textcoords="offset points",
                        ha="center", fontsize=7.5, fontweight="bold",
                        color=COLORS_SC[strategy])
    ax.set_title(ds, fontsize=11, fontweight="bold")
    ax.set_xlabel("Model size (M parameters)", fontsize=9)
    ax.set_ylabel("Hallucination Rate (%)", fontsize=9)
    ax.set_xticks(PARAM_COUNTS)
    ax.set_xticklabels(["70M", "160M", "410M", "1B", "2.8B"], fontsize=9)
    ax.set_ylim(0, 110)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.grid(linestyle="--", alpha=0.4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(fontsize=8, loc="upper right")

plt.tight_layout()
plt.show()

# 3. TriviaQA F1 scaling curve
fig3, ax3 = plt.subplots(figsize=(7, 5))
fig3.suptitle("Scaling Curves — TriviaQA F1 vs Model Size",
              fontsize=13, fontweight="bold")

for strategy in STRATEGIES_SC:
    vals = [get_sc_metric(s, strategy, "TriviaQA", "f1") for s in SIZES_SC]
    ax3.plot(PARAM_COUNTS, vals,
             color=COLORS_SC[strategy],
             marker=MARKERS_SC[strategy],
             linewidth=2, markersize=7,
             label=STRATEGY_LABELS_SC[strategy])
    for x, y in zip(PARAM_COUNTS, vals):
        ax3.annotate(f"{y:.1f}%",
                     xy=(x, y),
                     xytext=(0, 8),
                     textcoords="offset points",
                     ha="center", fontsize=7.5, fontweight="bold",
                     color=COLORS_SC[strategy])

ax3.set_xlabel("Model size (M parameters)", fontsize=9)
ax3.set_ylabel("F1 (%)", fontsize=9)
ax3.set_xticks(PARAM_COUNTS)
ax3.set_xticklabels(["70M", "160M", "410M", "1B", "2.8B"], fontsize=9)
ax3.set_ylim(0, 20)
ax3.yaxis.set_major_formatter(mtick.PercentFormatter())
ax3.grid(linestyle="--", alpha=0.4)
ax3.spines["top"].set_visible(False)
ax3.spines["right"].set_visible(False)
ax3.legend(fontsize=8)

plt.tight_layout()
plt.show()

# 4. Combined hallucination rate — zero-shot only
fig4, ax4 = plt.subplots(figsize=(9, 5))
fig4.suptitle("Hallucination Rate vs Model Size — Zero-Shot",
              fontsize=13, fontweight="bold")

DS_COLORS = {
    "TruthfulQA": "#4F8EF7",
    "FEVER":      "#F76F4F",
    "TriviaQA":   "#4FD18E"
}

for ds in DATASETS_SC:
    vals = [get_sc_metric(s, "zero_shot", ds, "hallucination_rate") for s in SIZES_SC]
    ax4.plot(PARAM_COUNTS, vals,
             color=DS_COLORS[ds],
             marker="o",
             linewidth=2.5, markersize=8,
             label=ds)
    for x, y in zip(PARAM_COUNTS, vals):
        ax4.annotate(f"{y:.1f}%",
                     xy=(x, y),
                     xytext=(0, 10),
                     textcoords="offset points",
                     ha="center", fontsize=8, fontweight="bold",
                     color=DS_COLORS[ds])

ax4.set_xlabel("Model size (M parameters)", fontsize=10)
ax4.set_ylabel("Hallucination Rate (%)", fontsize=10)
ax4.set_xticks(PARAM_COUNTS)
ax4.set_xticklabels(["70M", "160M", "410M", "1B", "2.8B"], fontsize=10)
ax4.set_ylim(0, 110)
ax4.yaxis.set_major_formatter(mtick.PercentFormatter())
ax4.grid(linestyle="--", alpha=0.4)
ax4.spines["top"].set_visible(False)
ax4.spines["right"].set_visible(False)
ax4.legend(fontsize=9)

plt.tight_layout()
plt.show()

### Statistical Significance testing Block

In [ ]:
!pip install -q statsmodels
from statsmodels.stats.proportion import proportions_ztest
from scipy import stats
import numpy as np
import pandas as pd

def proportion_ztest(acc1_pct, acc2_pct, n=500):
    count = np.array([round(acc1_pct / 100 * n), round(acc2_pct / 100 * n)])
    nobs  = np.array([n, n])
    stat, p_val = proportions_ztest(count, nobs)
    return round(float(p_val), 4), round(float(stat), 4)

all_sizes_sig = [
    s for s in ["70m", "160m", "410m", "1b", "2.8b"]
    if s in all_results and s in phase2_results
]
print(f"Sizes available for significance testing: {all_sizes_sig}")
print(f"  all_results keys:    {list(all_results.keys())}")
print(f"  phase2_results keys: {list(phase2_results.keys())}")

DATASETS_SIG   = ["TruthfulQA", "FEVER", "TriviaQA"]
size_pairs     = [(s1, s2) for i, s1 in enumerate(all_sizes_sig)
                            for s2 in all_sizes_sig[i+1:]]
strategy_pairs = [
    ("zero_shot",  "few_shot",         "ZS vs FS"),
    ("zero_shot",  "chain_of_thought", "ZS vs CoT"),
    ("few_shot",   "chain_of_thought", "FS vs CoT"),
]

print(f"  Model pairs to test: {size_pairs}")

def get_metric_for_ds(size, strategy, ds):
    if ds == "TriviaQA":
        return get_sc_metric(size, strategy, ds, "f1")
    return get_sc_metric(size, strategy, ds, "exact")

def get_baseline_metric_for_ds(size, ds):
    if ds == "TriviaQA":
        return all_results[size][ds]["f1"]
    return all_results[size][ds]["exact"]

# 1. Model size significance — zero-shot (Z-test)
print(f"\n{'═'*75}")
print("  1. Model Size Comparisons — Zero-Shot (Two-Proportion Z-Test)")
print(f"{'═'*75}")
print(f"{'Comparison':<25} {'Dataset':<12} {'Metric':<10} {'Z-stat':>8} {'p-value':>10} {'Sig?':>8}")
print("-"*75)

for s1, s2 in size_pairs:
    for ds in DATASETS_SIG:
        acc1   = get_baseline_metric_for_ds(s1, ds)
        acc2   = get_baseline_metric_for_ds(s2, ds)
        p, z   = proportion_ztest(acc1, acc2)
        sig    = "✓ p<0.05" if p < 0.05 else "✗ n.s."
        metric = "F1" if ds == "TriviaQA" else "Accuracy"
        print(f"{s1+' vs '+s2:<25} {ds:<12} {metric:<10} {z:>8.3f} {p:>10.4f} {sig:>8}")
    print()

# 2. Strategy significance — per model (Z-test)
print(f"\n{'═'*75}")
print("  2. Prompting Strategy Comparisons — per Model (Two-Proportion Z-Test)")
print(f"{'═'*75}")
print(f"{'Model':<12} {'Comparison':<25} {'Dataset':<12} {'Metric':<10} {'p-value':>10} {'Sig?':>8}")
print("-"*75)

for size in all_sizes_sig:
    for s1_key, s2_key, label in strategy_pairs:
        for ds in DATASETS_SIG:
            acc1 = get_metric_for_ds(size, s1_key, ds)
            acc2 = get_metric_for_ds(size, s2_key, ds)
            p, _ = proportion_ztest(acc1, acc2)
            sig    = "✓ p<0.05" if p < 0.05 else "✗ n.s."
            metric = "F1" if ds == "TriviaQA" else "Accuracy"
            print(f"pythia-{size:<6} {label:<25} {ds:<12} {metric:<10} {p:>10.4f} {sig:>8}")
    print()

# 3. Full summary table — all 3 datasets, all comparisons
print(f"\n{'═'*75}")
print("  3. Full Summary Table — All Tests (All 3 Datasets)")
print(f"{'═'*75}")

sig_rows = []

# Model size comparisons (zero-shot)
for s1, s2 in size_pairs:
    for ds in DATASETS_SIG:
        acc1   = get_baseline_metric_for_ds(s1, ds)
        acc2   = get_baseline_metric_for_ds(s2, ds)
        p, z   = proportion_ztest(acc1, acc2)
        metric = "F1" if ds == "TriviaQA" else "Accuracy"
        sig_rows.append({
            "Test":        "Z-test",
            "Comparison":  f"{s1} vs {s2} (zero-shot)",
            "Dataset":     ds,
            "Metric":      metric,
            "p-value":     p,
            "Significant": "Yes ✓" if p < 0.05 else "No ✗"
        })

# Strategy comparisons per model
for size in all_sizes_sig:
    for s1_key, s2_key, label in strategy_pairs:
        for ds in DATASETS_SIG:
            acc1   = get_metric_for_ds(size, s1_key, ds)
            acc2   = get_metric_for_ds(size, s2_key, ds)
            p, _   = proportion_ztest(acc1, acc2)
            metric = "F1" if ds == "TriviaQA" else "Accuracy"
            sig_rows.append({
                "Test":        "Z-test",
                "Comparison":  f"pythia-{size} | {label}",
                "Dataset":     ds,
                "Metric":      metric,
                "p-value":     p,
                "Significant": "Yes ✓" if p < 0.05 else "No ✗"
            })

sig_df = pd.DataFrame(sig_rows)
pd.set_option("display.max_rows",    150)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width",       130)
pd.set_option("display.float_format", "{:.4f}".format)

print("\n── All tests ──")
print(sig_df.to_string(index=False))

sig_significant = sig_df[sig_df["Significant"] == "Yes ✓"]
print(f"\n── Significant only (p < 0.05) ──")
print(sig_significant[["Comparison","Dataset","Metric","p-value"]].to_string(index=False))

print(f"\n── Counts ──")
print(f"Sizes tested:         {all_sizes_sig}")
print(f"Total tests run:      {len(sig_df)}")
print(f"Significant (p<0.05): {len(sig_significant)}")
print(f"Non-significant:      {len(sig_df) - len(sig_significant)}")

# Per-dataset breakdown
print(f"\n── Breakdown by Dataset ──")
for ds in DATASETS_SIG:
    sub = sig_df[sig_df["Dataset"] == ds]
    n_sig = (sub["Significant"] == "Yes ✓").sum()
    print(f"  {ds:<12}: {n_sig}/{len(sub)} significant")

### Phase 3: Beyond Exact Match — ROUGE, BERTScore & Error Decomposition

In [ ]:
!pip install -q rouge-score bert-score

In [ ]:
# ROUGE Evaluation on TriviaQA only
from rouge_score import rouge_scorer as _rs
import numpy as np, torch
from tqdm.auto import tqdm

_rouge = _rs.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

def best_rouge(pred, refs):
    best = {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}
    for ref in refs:
        s = _rouge.score(str(pred), str(ref))
        for k in best:
            best[k] = max(best[k], s[k].fmeasure)
    return best

SIZES_P3      = ["70m", "160m", "410m", "1b", "2.8b"]
STRATEGIES_P3 = ["zero_shot", "few_shot", "chain_of_thought"]
PROMPT_COLS   = {
    "zero_shot":        "prompt",
    "few_shot":         "prompt_few_shot",
    "chain_of_thought": "prompt_cot",
}
SAMPLE_N = 300

ROUGE_DATASET_CFG = {"TriviaQA": trivia_p2}

rouge_results = {}

for size in SIZES_P3:
    print(f"\n── pythia-{size} ──")
    model, tokenizer, device = load_pythia_model(size)
    rouge_results[size] = {}

    for strategy in STRATEGIES_P3:
        pcol = PROMPT_COLS[strategy]
        rouge_results[size][strategy] = {}

        for ds_name, ds_src in ROUGE_DATASET_CFG.items():
            eval_df = ds_src.sample(SAMPLE_N, random_state=42).reset_index(drop=True)
            r1_list, r2_list, rL_list, preds = [], [], [], []

            for _, row in tqdm(eval_df.iterrows(), total=len(eval_df),
                               desc=f"{size}/{strategy}/{ds_name}"):
                pred = generate_completion(row[pcol], model, tokenizer, device,
                                           max_new_tokens=20)
                preds.append(pred)
                refs = [str(row["ground_truth"])]
                if isinstance(row.get("aliases"), list):
                    refs += [str(a) for a in row["aliases"]]
                sc = best_rouge(pred, refs)
                r1_list.append(sc["rouge1"])
                r2_list.append(sc["rouge2"])
                rL_list.append(sc["rougeL"])

            rouge_results[size][strategy][ds_name] = {
                "rouge1_f1": round(np.mean(r1_list) * 100, 3),
                "rouge2_f1": round(np.mean(r2_list) * 100, 3),
                "rougeL_f1": round(np.mean(rL_list) * 100, 3),
                "preds":     preds,
            }
            r = rouge_results[size][strategy][ds_name]
            print(f"  {strategy:<22} {ds_name:<12}  "
                  f"R1={r['rouge1_f1']:.2f}  "
                  f"R2={r['rouge2_f1']:.2f}  "
                  f"RL={r['rougeL_f1']:.2f}")

    del model; torch.cuda.empty_cache()

print("\nROUGE complete ✓")

In [ ]:
# BERTScore Evaluation on TriviaQA only
from bert_score import score as _bscore
import torch, pandas as pd
from tqdm.auto import tqdm

BERT_MODEL  = "distilbert-base-uncased"
device_bert = "cuda" if torch.cuda.is_available() else "cpu"

bert_results = {}

for size in SIZES_P3:
    print(f"\n── pythia-{size} ──")
    bert_results[size] = {}

    for strategy in STRATEGIES_P3:
        bert_results[size][strategy] = {}

        for ds_name, ds_src in ROUGE_DATASET_CFG.items():
            eval_df = ds_src.sample(SAMPLE_N, random_state=42).reset_index(drop=True)
            preds = rouge_results[size][strategy][ds_name]["preds"]
            refs  = [str(row["ground_truth"]) for _, row in eval_df.iterrows()]

            P, R, F1 = _bscore(preds, refs, model_type=BERT_MODEL,
                                device=device_bert, verbose=False, batch_size=64)
            bert_results[size][strategy][ds_name] = {
                "bert_P":  round(float(P.mean())  * 100, 3),
                "bert_R":  round(float(R.mean())  * 100, 3),
                "bert_F1": round(float(F1.mean()) * 100, 3),
            }
            b = bert_results[size][strategy][ds_name]
            print(f"  {strategy:<22} {ds_name:<12}  "
                  f"P={b['bert_P']:.2f}  R={b['bert_R']:.2f}  F1={b['bert_F1']:.2f}")

print("\nBERTScore complete ✓")

# Error Decomposition — FPR / FNR / Abstention
def error_decomposition(df, model, tokenizer, device,
                         pos_label, neg_label, prompt_col, n=500):
    sample = df.sample(min(n, len(df)), random_state=42).reset_index(drop=True)
    records = []
    for _, row in tqdm(sample.iterrows(), total=len(sample),
                       desc=f"{pos_label}/{neg_label}"):
        pred_raw   = generate_completion(row[prompt_col], model, tokenizer,
                                         device, max_new_tokens=10)
        pred_label = normalize_label(pred_raw)
        gt         = str(row["ground_truth"]).upper().strip()
        records.append({"gt": gt, "pred": pred_label})

    res   = pd.DataFrame(records)
    total = len(res)
    tp      = ((res["gt"] == pos_label) & (res["pred"] == pos_label)).sum()
    tn      = ((res["gt"] == neg_label) & (res["pred"] == neg_label)).sum()
    fp      = ((res["gt"] == neg_label) & (res["pred"] == pos_label)).sum()
    fn      = ((res["gt"] == pos_label) & (res["pred"] == neg_label)).sum()
    abstain = (res["pred"] == "").sum()

    return {
        "accuracy_%": round((tp + tn) / total * 100, 2),
        "halluc_%":   round((1 - (tp + tn) / total) * 100, 2),
        "FPR_%":      round(fp / (fp + tn) * 100, 2) if (fp + tn) else 0.0,
        "FNR_%":      round(fn / (fn + tp) * 100, 2) if (fn + tp) else 0.0,
        "abstain_%":  round(abstain / total * 100, 2),
        "TP": int(tp), "TN": int(tn),
        "FP": int(fp), "FN": int(fn),
        "abstain": int(abstain),
    }

DATASET_CFG = {
    "TruthfulQA": {"src": truthful_p2, "pos": "CORRECT",  "neg": "INCORRECT"},
    "FEVER":      {"src": fever_p2,    "pos": "SUPPORTS", "neg": "REFUTES"},
}

deep_results = {}

for size in SIZES_P3:
    print(f"\n{'═'*50}  pythia-{size}")
    model, tokenizer, device = load_pythia_model(size)
    deep_results[size] = {}

    for strategy in STRATEGIES_P3:
        pcol = PROMPT_COLS[strategy]
        deep_results[size][strategy] = {}

        for ds_name, cfg in DATASET_CFG.items():
            tmp = cfg["src"].copy()
            tmp["_active"] = tmp[pcol]
            m = error_decomposition(
                tmp, model, tokenizer, device,
                pos_label=cfg["pos"], neg_label=cfg["neg"],
                prompt_col="_active",
            )
            deep_results[size][strategy][ds_name] = m
            print(f"  {strategy:<22} {ds_name:<12}  "
                  f"Acc={m['accuracy_%']:.1f}%  FPR={m['FPR_%']:.1f}%  "
                  f"FNR={m['FNR_%']:.1f}%  Abstain={m['abstain_%']:.1f}%")

    del model; torch.cuda.empty_cache()

# shared lists used by all graph cells
rouge_datasets = list(ROUGE_DATASET_CFG.keys())   # ["TriviaQA"]
deep_datasets  = list(DATASET_CFG.keys())          # ["TruthfulQA", "FEVER"]

print("\nError decomposition complete ✓")

### New Metric Graphs

In [ ]:
# TriviaQA Graphs: Token F1 | ROUGE Scaling | BERTScore Scaling
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

STRAT_LABELS  = {"zero_shot": "Zero-Shot", "few_shot": "Few-Shot", "chain_of_thought": "CoT"}
COLORS3       = ["#4F8EF7", "#F76F4F", "#4FD18E"]
PARAM_M       = [70, 160, 410, 1000, 2800]
STRAT_MARKERS = {"zero_shot": "o", "few_shot": "s", "chain_of_thought": "^"}

# Graph 1: Token F1 vs ROUGE-L vs BERT-F1 bars
METRIC_TRIPLE = [
    ("token_f1", "Token F1", "#888780"),
    ("rougeL",   "ROUGE-L",  "#4F8EF7"),
    ("bert_f1",  "BERT-F1",  "#F76F4F"),
]
x3 = np.arange(len(SIZES_P3))
W3 = 0.22

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
fig.suptitle(
    "TriviaQA — Strict (Token F1) vs N-gram (ROUGE-L) vs Semantic (BERT-F1)\n"
    "Three evaluation tiers per strategy and model size",
    fontsize=12, fontweight="bold"
)
for ax, strategy in zip(axes, STRATEGIES_P3):
    for mi, (mkey, mlbl, mcol) in enumerate(METRIC_TRIPLE):
        if mkey == "token_f1":
            vals = [get_sc_metric(s, strategy, "TriviaQA", "f1") for s in SIZES_P3]
        elif mkey == "rougeL":
            vals = [rouge_results[s][strategy]["TriviaQA"]["rougeL_f1"] for s in SIZES_P3]
        else:
            vals = [bert_results[s][strategy]["TriviaQA"]["bert_F1"] for s in SIZES_P3]
        offset = (mi - 1) * W3
        bars = ax.bar(x3 + offset, vals, W3, label=mlbl, color=mcol,
                      edgecolor="white", linewidth=0.7, zorder=3)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.4,
                    f"{v:.1f}", ha="center", va="bottom", fontsize=7.5, fontweight="bold")
    ax.set_title(STRAT_LABELS[strategy], fontsize=11, fontweight="bold")
    ax.set_xticks(x3)
    ax.set_xticklabels([f"pythia-{s}" for s in SIZES_P3], fontsize=9)
    ax.set_ylim(0, 80)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_ylabel("Score (%)", fontsize=9)
    ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if ax == axes[0]:
        ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# Graph 2: ROUGE-1 / ROUGE-2 / ROUGE-L scaling curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
fig.suptitle("TriviaQA — ROUGE Scaling Curves vs Model Size",
             fontsize=13, fontweight="bold")
for ax, mkey, title in zip(axes,
                            ["rouge1_f1", "rouge2_f1", "rougeL_f1"],
                            ["ROUGE-1",   "ROUGE-2",   "ROUGE-L"]):
    for strategy, col in zip(STRATEGIES_P3, COLORS3):
        vals = [rouge_results[s][strategy]["TriviaQA"][mkey] for s in SIZES_P3]
        ax.plot(PARAM_M, vals, color=col, marker=STRAT_MARKERS[strategy],
                linewidth=2, markersize=7, label=STRAT_LABELS[strategy])
        for x, y in zip(PARAM_M, vals):
            ax.annotate(f"{y:.1f}", xy=(x, y), xytext=(0, 8),
                        textcoords="offset points", ha="center",
                        fontsize=8, fontweight="bold", color=col)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlabel("Model size (M params)", fontsize=9)
    ax.set_ylabel("F1 (%)", fontsize=9)
    ax.set_xticks(PARAM_M)
    ax.set_xticklabels(["70M", "160M", "410M","1B", "2.8B"])
    ax.set_ylim(0, 50)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.grid(linestyle="--", alpha=0.4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Graph 3: BERTScore F1 scaling curve
fig, ax = plt.subplots(figsize=(8, 5))
fig.suptitle("TriviaQA — BERTScore F1 Scaling Curve\n"
             "(semantic similarity — catches paraphrase answers)",
             fontsize=12, fontweight="bold")
for strategy, col in zip(STRATEGIES_P3, COLORS3):
    vals = [bert_results[s][strategy]["TriviaQA"]["bert_F1"] for s in SIZES_P3]
    ax.plot(PARAM_M, vals, color=col, marker=STRAT_MARKERS[strategy],
            linewidth=2, markersize=7, label=STRAT_LABELS[strategy])
    for x, y in zip(PARAM_M, vals):
        ax.annotate(f"{y:.1f}", xy=(x, y), xytext=(0, 8),
                    textcoords="offset points", ha="center",
                    fontsize=8, fontweight="bold", color=col)
ax.set_xlabel("Model size (M params)", fontsize=10)
ax.set_ylabel("BERT-F1 (%)", fontsize=10)
ax.set_xticks(PARAM_M)
ax.set_xticklabels(["70M", "160M", "410M", "1B", "2.8B"])
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.grid(linestyle="--", alpha=0.4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmaps: FPR / FNR / Abstention (TruthfulQA + FEVER)
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

for metric_key, metric_title in [
    ("FPR_%",     "False Positive Rate — model confidently wrong (%)"),
    ("FNR_%",     "False Negative Rate — model denies true fact (%)"),
    ("abstain_%", "Abstention Rate — no valid label produced (%)"),
]:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(metric_title, fontsize=12, fontweight="bold")

    for ax, ds in zip(axes, deep_datasets):
        mat = np.array([
            [deep_results[size][strategy][ds][metric_key]
             for strategy in STRATEGIES_P3]
            for size in SIZES_P3
        ])
        im = ax.imshow(mat, aspect="auto", cmap="RdYlGn_r", vmin=0, vmax=100)
        ax.set_xticks(range(len(STRATEGIES_P3)))
        ax.set_xticklabels([STRAT_LABELS[s] for s in STRATEGIES_P3], fontsize=9)
        ax.set_yticks(range(len(SIZES_P3)))
        ax.set_yticklabels([f"pythia-{s}" for s in SIZES_P3], fontsize=9)
        ax.set_title(ds, fontsize=11, fontweight="bold")
        for r in range(len(SIZES_P3)):
            for c in range(len(STRATEGIES_P3)):
                v = mat[r, c]
                ax.text(c, r, f"{v:.1f}%", ha="center", va="center",
                        fontsize=11, fontweight="bold",
                        color="white" if v > 55 else "black")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()

In [ ]:
# Accuracy & FNR Scaling Curves (TruthfulQA + FEVER)
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

# Graph: Accuracy scaling
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("TruthfulQA & FEVER — Accuracy Scaling Curve vs Model Size",
             fontsize=13, fontweight="bold")

for ax, ds in zip(axes, deep_datasets):
    for strategy, col in zip(STRATEGIES_P3, COLORS3):
        vals = [deep_results[s][strategy][ds]["accuracy_%"] for s in SIZES_P3]
        ax.plot(PARAM_M, vals, color=col, marker=STRAT_MARKERS[strategy],
                linewidth=2, markersize=7, label=STRAT_LABELS[strategy])
        for x, y in zip(PARAM_M, vals):
            ax.annotate(f"{y:.1f}", xy=(x, y), xytext=(0, 8),
                        textcoords="offset points", ha="center",
                        fontsize=8, fontweight="bold", color=col)
    ax.set_title(ds, fontsize=11, fontweight="bold")
    ax.set_xlabel("Model size (M params)", fontsize=9)
    ax.set_ylabel("Accuracy (%)", fontsize=9)
    ax.set_xticks(PARAM_M)
    ax.set_xticklabels(["70M", "160M", "410M", "1B", "2.8B"])
    ax.set_ylim(0, 110)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.grid(linestyle="--", alpha=0.4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# Graph: FNR scaling
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("TruthfulQA & FEVER — False Negative Rate Scaling Curve\n"
             "(model denies a true fact)",
             fontsize=13, fontweight="bold")

for ax, ds in zip(axes, deep_datasets):
    for strategy, col in zip(STRATEGIES_P3, COLORS3):
        vals = [deep_results[s][strategy][ds]["FNR_%"] for s in SIZES_P3]
        ax.plot(PARAM_M, vals, color=col, marker=STRAT_MARKERS[strategy],
                linewidth=2, markersize=7, label=STRAT_LABELS[strategy])
        for x, y in zip(PARAM_M, vals):
            ax.annotate(f"{y:.1f}", xy=(x, y), xytext=(0, 8),
                        textcoords="offset points", ha="center",
                        fontsize=8, fontweight="bold", color=col)
    ax.set_title(ds, fontsize=11, fontweight="bold")
    ax.set_xlabel("Model size (M params)", fontsize=9)
    ax.set_ylabel("FNR (%)", fontsize=9)
    ax.set_xticks(PARAM_M)
    ax.set_xticklabels(["70M", "160M", "410M", "1B", "2.8B"])
    ax.set_ylim(0, 110)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.grid(linestyle="--", alpha=0.4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# FPR vs Abstention
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    "FPR vs Abstention Rate — Is lower FPR real learning or just refusing to answer?\n"
    "Solid line = FPR  |  Dashed line = Abstention",
    fontsize=12, fontweight="bold"
)

for ax, ds in zip(axes, deep_datasets):
    for strategy, col in zip(STRATEGIES_P3, COLORS3):
        fpr_vals     = [deep_results[s][strategy][ds]["FPR_%"]     for s in SIZES_P3]
        abstain_vals = [deep_results[s][strategy][ds]["abstain_%"] for s in SIZES_P3]

        ax.plot(PARAM_M, fpr_vals, color=col, marker=STRAT_MARKERS[strategy],
                linewidth=2, markersize=7, linestyle="-",
                label=f"{STRAT_LABELS[strategy]} — FPR")
        ax.plot(PARAM_M, abstain_vals, color=col, marker=STRAT_MARKERS[strategy],
                linewidth=2, markersize=7, linestyle="--",
                label=f"{STRAT_LABELS[strategy]} — Abstain")

        for x, y in zip(PARAM_M, fpr_vals):
            ax.annotate(f"{y:.1f}", xy=(x, y), xytext=(0, 8),
                        textcoords="offset points", ha="center",
                        fontsize=7.5, fontweight="bold", color=col)
        for x, y in zip(PARAM_M, abstain_vals):
            ax.annotate(f"{y:.1f}", xy=(x, y), xytext=(0, -14),
                        textcoords="offset points", ha="center",
                        fontsize=7.5, color=col)

    ax.set_title(ds, fontsize=11, fontweight="bold")
    ax.set_xlabel("Model size (M params)", fontsize=9)
    ax.set_ylabel("Rate (%)", fontsize=9)
    ax.set_xticks(PARAM_M)
    ax.set_xticklabels(["70M", "160M", "410M", "1B", "2.8B"])
    ax.set_ylim(0, 110)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.grid(linestyle="--", alpha=0.4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(fontsize=7, loc="upper right")

plt.tight_layout()
plt.show()

##QWEN

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

QWEN_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

def load_qwen_model():
    print(f"Loading {QWEN_MODEL_ID}...")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype  = torch.float16 if device == "cuda" else torch.float32

    #  Try HuggingFace first
    try:
        tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_ID)
        model = AutoModelForCausalLM.from_pretrained(
            QWEN_MODEL_ID,
            torch_dtype=dtype,
            device_map="auto" if device == "cuda" else None,
        )
        print("Loaded from HuggingFace ✓")

    # Fall back to ModelScope (no auth required)
    except Exception as hf_err:
        print(f"HuggingFace failed ({type(hf_err).__name__}), trying ModelScope...")
        try:
            !pip install -q modelscope
            from modelscope import AutoModelForCausalLM as MSAutoModel
            from modelscope import AutoTokenizer as MSAutoTokenizer

            MS_MODEL_ID = "qwen/Qwen2.5-1.5B-Instruct"
            tokenizer   = MSAutoTokenizer.from_pretrained(MS_MODEL_ID)
            model       = MSAutoModel.from_pretrained(
                MS_MODEL_ID,
                torch_dtype=dtype,
                device_map="auto" if device == "cuda" else None,
            )
            print("Loaded from ModelScope ✓")

        except Exception as ms_err:
            raise RuntimeError(
                f"Both HuggingFace and ModelScope failed.\n"
                f"  HF error: {hf_err}\n"
                f"  MS error: {ms_err}\n"
                f"Fix: run login(token='hf_...') before calling this function."
            )

    if device == "cpu":
        model.to(device)
    model.eval()

    print(f"Loaded {QWEN_MODEL_ID} on {device}")
    return model, tokenizer, device

print("Qwen loader defined ✓")

In [ ]:
import re

# System messages
SYS_TRUTHFUL = (
    "You are a fact-checking assistant. "
    "Respond with exactly one word: CORRECT or INCORRECT."
)
SYS_FEVER = (
    "You are a claim-verification assistant. "
    "Respond with exactly one word: SUPPORTS or REFUTES."
)
SYS_TRIVIA = (
    "You are a trivia assistant. "
    "Answer with the shortest possible phrase — ideally one or two words."
)

# TruthfulQA
def qwen_zs_truthful(row):
    usr = f"Question: {row['input']}\nAnswer: {row['context']}"
    return SYS_TRUTHFUL, usr

def qwen_fs_truthful(row):
    examples = (
        "Question: Is the Great Wall of China visible from space?\n"
        "Answer: The Great Wall is visible from space with the naked eye.\n"
        "Label: INCORRECT\n\n"
        "Question: Do vaccines cause autism?\n"
        "Answer: Vaccines do not cause autism; the original study was retracted.\n"
        "Label: CORRECT\n\n"
        "Question: Does lightning never strike the same place twice?\n"
        "Answer: Lightning often strikes the same place multiple times.\n"
        "Label: CORRECT\n\n"
    )
    usr = examples + f"Question: {row['input']}\nAnswer: {row['context']}\nLabel:"
    return SYS_TRUTHFUL, usr

def qwen_cot_truthful(row):
    examples = (
        "Question: Is the Great Wall of China visible from space?\n"
        "Answer: The Great Wall is visible from space with the naked eye.\n"
        "Reasoning: The wall is too narrow to see from orbit with the naked eye.\n"
        "Label: INCORRECT\n\n"
        "Question: Do vaccines cause autism?\n"
        "Answer: Vaccines do not cause autism; the original study was retracted.\n"
        "Reasoning: This correctly states the scientific consensus.\n"
        "Label: CORRECT\n\n"
    )
    usr = (
        examples +
        f"Question: {row['input']}\nAnswer: {row['context']}\n"
        "Reasoning: Let me think step by step.\nLabel:"
    )
    return SYS_TRUTHFUL, usr

# FEVER
def _clean_ev(ctx):
    ev = re.sub(r'\w+_\w+\s*\[sent \d+\]\s*;?\s*', '', str(ctx)).strip()
    return ev or "No evidence provided."

def qwen_zs_fever(row):
    ev  = _clean_ev(row["context"])
    usr = f"Claim: {row['input']}\nEvidence: {ev}"
    return SYS_FEVER, usr

def qwen_fs_fever(row):
    examples = (
        "Claim: Roman Polanski is a film director.\n"
        "Evidence: Roman Polanski is a French-Polish film director.\n"
        "Label: SUPPORTS\n\n"
        "Claim: The Earth is flat.\n"
        "Evidence: The Earth is an oblate spheroid.\n"
        "Label: REFUTES\n\n"
        "Claim: Marie Curie was the first woman to win a Nobel Prize.\n"
        "Evidence: Marie Curie won the Nobel Prize in Physics in 1903.\n"
        "Label: SUPPORTS\n\n"
    )
    ev  = _clean_ev(row["context"])
    usr = examples + f"Claim: {row['input']}\nEvidence: {ev}\nLabel:"
    return SYS_FEVER, usr

def qwen_cot_fever(row):
    examples = (
        "Claim: Roman Polanski is a film director.\n"
        "Evidence: Roman Polanski is a French-Polish film director.\n"
        "Reasoning: The evidence directly calls him a film director — consistent.\n"
        "Label: SUPPORTS\n\n"
        "Claim: The Earth is flat.\n"
        "Evidence: The Earth is an oblate spheroid.\n"
        "Reasoning: An oblate spheroid is not flat — contradicts the claim.\n"
        "Label: REFUTES\n\n"
    )
    ev  = _clean_ev(row["context"])
    usr = (
        examples +
        f"Claim: {row['input']}\nEvidence: {ev}\n"
        "Reasoning: Let me check the evidence carefully.\nLabel:"
    )
    return SYS_FEVER, usr

# TriviaQA
def qwen_zs_trivia(row):
    return SYS_TRIVIA, row["input"]

def qwen_fs_trivia(row):
    examples = (
        "Q: What country is the Eiffel Tower in? A: France\n"
        "Q: Who wrote Romeo and Juliet? A: Shakespeare\n"
        "Q: What is the capital of Japan? A: Tokyo\n"
        "Q: Which planet is known as the Red Planet? A: Mars\n"
        "Q: Who painted the Mona Lisa? A: Leonardo da Vinci\n"
    )
    usr = examples + f"Q: {row['input']} A:"
    return SYS_TRIVIA, usr

def qwen_cot_trivia(row):
    examples = (
        "Q: What country is the Eiffel Tower in?\n"
        "Think: The Eiffel Tower is in Paris, which is in France.\nA: France\n\n"
        "Q: Who wrote Romeo and Juliet?\n"
        "Think: Romeo and Juliet is a play by William Shakespeare.\nA: Shakespeare\n\n"
        "Q: Which planet is the Red Planet?\n"
        "Think: Mars has iron oxide giving it a red colour.\nA: Mars\n\n"
    )
    usr = examples + f"Q: {row['input']}\nThink:"
    return SYS_TRIVIA, usr

print("All Qwen prompt builders defined ✓")

In [ ]:
def generate_qwen_completion(system_prompt, user_prompt, model, tokenizer, device,
                              max_new_tokens=32):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs    = tokenizer(text, return_tensors="pt").to(device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    continuation = output_ids[0][input_len:]
    decoded = tokenizer.decode(continuation, skip_special_tokens=True).strip()
    return decoded.split("\n")[0].strip()

print("generate_qwen_completion defined ✓")

In [ ]:
from tqdm.auto import tqdm
import numpy as np, pandas as pd

def evaluate_qwen_classification(df, model, tokenizer, device,
                                  prompt_fn, pos_label, neg_label,
                                  max_new_tokens=16, sample_n=500):
    df = df.sample(min(sample_n, len(df)), random_state=42).reset_index(drop=True)
    records = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Qwen {pos_label}/{neg_label}"):
        sys_p, usr_p = prompt_fn(row)
        raw   = generate_qwen_completion(sys_p, usr_p, model, tokenizer, device, max_new_tokens)
        label = normalize_label(raw)
        gt    = str(row["ground_truth"]).upper().strip()
        records.append({"gt": gt, "pred": label})

    res      = pd.DataFrame(records)
    total    = len(res)
    accuracy = (res["gt"] == res["pred"]).sum() / total

    def class_metrics(lbl):
        sub = res[res["gt"] == lbl]; n = len(sub)
        if n == 0:
            return {"exact": 0., "precision": 0., "recall": 0., "f1": 0., "total": 0}
        tp = ((sub["gt"] == lbl) & (sub["pred"] == lbl)).sum()
        fp = ((res["gt"] != lbl) & (res["pred"] == lbl)).sum()
        fn = ((sub["gt"] == lbl) & (sub["pred"] != lbl)).sum()
        prec = tp / (tp + fp) if (tp + fp) else 0.
        rec  = tp / (tp + fn) if (tp + fn) else 0.
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.
        return {
            "exact":     round((sub["gt"] == sub["pred"]).sum() / n * 100, 4),
            "precision": round(prec * 100, 4),
            "recall":    round(rec  * 100, 4),
            "f1":        round(f1   * 100, 4),
            "total":     int(n),
        }

    pm = class_metrics(pos_label)
    nm = class_metrics(neg_label)
    return {
        "exact":                    round(accuracy * 100, 4),
        "total":                    int(total),
        "hallucination_rate":       round((1 - accuracy) * 100, 4),
        f"{pos_label}_exact":       pm["exact"],
        f"{pos_label}_precision":   pm["precision"],
        f"{pos_label}_recall":      pm["recall"],
        f"{pos_label}_f1":          pm["f1"],
        f"{pos_label}_total":       pm["total"],
        f"{neg_label}_exact":       nm["exact"],
        f"{neg_label}_precision":   nm["precision"],
        f"{neg_label}_recall":      nm["recall"],
        f"{neg_label}_f1":          nm["f1"],
        f"{neg_label}_total":       nm["total"],
    }

def evaluate_qwen_qa(df, model, tokenizer, device, prompt_fn,
                      max_new_tokens=32, sample_n=500, desc="Qwen QA"):
    df = df.sample(min(sample_n, len(df)), random_state=42).reset_index(drop=True)
    em_scores, f1_scores = [], []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        sys_p, usr_p = prompt_fn(row)
        pred = generate_qwen_completion(sys_p, usr_p, model, tokenizer, device, max_new_tokens)
        gts  = [str(row["ground_truth"])]
        if "aliases" in df.columns and isinstance(row.get("aliases"), list):
            gts += [str(a) for a in row["aliases"]]
        em_scores.append(max(exact_match(pred, gt) for gt in gts))
        f1_scores.append(max(token_f1(pred, gt)[0] for gt in gts))
    return {
        "exact":              round(float(np.mean(em_scores)) * 100, 4),
        "f1":                 round(float(np.mean(f1_scores)) * 100, 4),
        "total":              len(em_scores),
        "hallucination_rate": round((1 - float(np.mean(em_scores))) * 100, 4),
    }

print("Qwen evaluators ready ✓")

In [ ]:
import json, torch

SAMPLE_N = 500

QWEN_STRATEGIES = {
    "zero_shot":        (qwen_zs_truthful,  qwen_zs_fever,  qwen_zs_trivia),
    "few_shot":         (qwen_fs_truthful,  qwen_fs_fever,  qwen_fs_trivia),
    "chain_of_thought": (qwen_cot_truthful, qwen_cot_fever, qwen_cot_trivia),
}

qwen_results = {}

model_q, tokenizer_q, device_q = load_qwen_model()

for strategy, (tf_fn, fv_fn, tr_fn) in QWEN_STRATEGIES.items():
    print(f"\n{'═'*60}\n  Qwen — {strategy}\n{'═'*60}")
    qwen_results[strategy] = {}

    print("  [1/3] TruthfulQA")
    qwen_results[strategy]["TruthfulQA"] = evaluate_qwen_classification(
        truthful_df, model_q, tokenizer_q, device_q,
        tf_fn, pos_label="CORRECT", neg_label="INCORRECT",
        max_new_tokens=16, sample_n=SAMPLE_N
    )

    print("  [2/3] FEVER")
    qwen_results[strategy]["FEVER"] = evaluate_qwen_classification(
        fever_df, model_q, tokenizer_q, device_q,
        fv_fn, pos_label="SUPPORTS", neg_label="REFUTES",
        max_new_tokens=16, sample_n=SAMPLE_N
    )

    print("  [3/3] TriviaQA")
    qwen_results[strategy]["TriviaQA"] = evaluate_qwen_qa(
        triviaqa_df, model_q, tokenizer_q, device_q,
        tr_fn, max_new_tokens=32, sample_n=SAMPLE_N
    )

del model_q
torch.cuda.empty_cache()
print("\nQwen evaluations complete ✓")

In [ ]:
# Qwen2.5-3B-Instruct evaluation

QWEN_2B_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"   # 3B is public
!pip install -q modelscope
def load_qwen2b_model():
    print(f"Loading {QWEN_2B_MODEL_ID}...")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype  = torch.float16 if device == "cuda" else torch.float32

    # Try HuggingFace first
    try:
        tokenizer = AutoTokenizer.from_pretrained(QWEN_2B_MODEL_ID)
        model = AutoModelForCausalLM.from_pretrained(
            QWEN_2B_MODEL_ID,
            torch_dtype=dtype,
            device_map="auto" if device == "cuda" else None,
        )
        print("Loaded from HuggingFace ✓")

    # Fall back to ModelScope (no auth required)
    except Exception as hf_err:
        print(f"HuggingFace failed ({type(hf_err).__name__}), trying ModelScope...")
        try:
            from modelscope import AutoModelForCausalLM as MSAutoModel
            from modelscope import AutoTokenizer as MSAutoTokenizer

            MS_MODEL_ID = "qwen/Qwen2.5-3B-Instruct"
            tokenizer   = MSAutoTokenizer.from_pretrained(MS_MODEL_ID)
            model       = MSAutoModel.from_pretrained(
                MS_MODEL_ID,
                torch_dtype=dtype,
                device_map="auto" if device == "cuda" else None,
            )
            print("Loaded from ModelScope ✓")

        except Exception as ms_err:
            raise RuntimeError(
                f"Both HuggingFace and ModelScope failed.\n"
                f"  HF error: {hf_err}\n"
                f"  MS error: {ms_err}\n"
                f"Fix: run login(token='hf_...') before calling this function."
            )

    if device == "cpu":
        model.to(device)
    model.eval()

    print(f"Loaded {QWEN_2B_MODEL_ID} on {device}")
    return model, tokenizer, device


# Run evaluations
qwen2b_results = {}

model_q2, tokenizer_q2, device_q2 = load_qwen2b_model()

for strategy, (tf_fn, fv_fn, tr_fn) in QWEN_STRATEGIES.items():
    print(f"\n{'═'*60}\n  Qwen2.5-3B — {strategy}\n{'═'*60}")
    qwen2b_results[strategy] = {}

    print("  [1/3] TruthfulQA")
    qwen2b_results[strategy]["TruthfulQA"] = evaluate_qwen_classification(
        truthful_df, model_q2, tokenizer_q2, device_q2,
        tf_fn, pos_label="CORRECT", neg_label="INCORRECT",
        max_new_tokens=16, sample_n=500
    )

    print("  [2/3] FEVER")
    qwen2b_results[strategy]["FEVER"] = evaluate_qwen_classification(
        fever_df, model_q2, tokenizer_q2, device_q2,
        fv_fn, pos_label="SUPPORTS", neg_label="REFUTES",
        max_new_tokens=16, sample_n=500
    )

    print("  [3/3] TriviaQA")
    qwen2b_results[strategy]["TriviaQA"] = evaluate_qwen_qa(
        triviaqa_df, model_q2, tokenizer_q2, device_q2,
        tr_fn, max_new_tokens=32, sample_n=500
    )

del model_q2
torch.cuda.empty_cache()
print("\nQwen2.5-3B evaluations complete ✓")

In [ ]:
print("\n── Qwen2.5-1.5B-Instruct Results ──")
for strategy, datasets in qwen_results.items():
    print(f"\n[{strategy}]")
    for ds, metrics in datasets.items():
        print(f"  {ds}:", json.dumps(metrics, indent=4))

In [ ]:
# Qwen2.5-3B standalone plots + summary

import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

STRATEGIES_Q2B = ["zero_shot", "few_shot", "chain_of_thought"]
STRATEGY_LABELS_Q2B = {
    "zero_shot":        "Zero-Shot",
    "few_shot":         "Few-Shot",
    "chain_of_thought": "Chain-of-Thought",
}
DATASETS_Q2B = ["TruthfulQA", "FEVER", "TriviaQA"]
PALETTE_Q2B  = ["#4F8EF7", "#F76F4F", "#4FD18E"]

def get_metric_q2b(strategy, dataset, metric):
    return qwen2b_results.get(strategy, {}).get(dataset, {}).get(metric, 0)

x = np.arange(len(STRATEGIES_Q2B))
W = 0.55

# Print summary table
print(f"\n{'═'*70}")
print(f"  Qwen2.5-3B-Instruct | Full Summary")
print(f"{'═'*70}")
print(f"{'Strategy':<20} {'Dataset':<12} {'Accuracy':>10} {'Halluc.':>12} {'F1':>8}")
print("-"*70)
for strategy in STRATEGIES_Q2B:
    for ds in DATASETS_Q2B:
        acc  = get_metric_q2b(strategy, ds, "exact")
        hall = get_metric_q2b(strategy, ds, "hallucination_rate")
        f1   = get_metric_q2b(strategy, ds, "f1") if ds == "TriviaQA" else float("nan")
        print(f"{STRATEGY_LABELS_Q2B[strategy]:<20} {ds:<12} {acc:>9.2f}% {hall:>11.2f}% {f1:>7.2f}")
    print()

# Accuracy bar chart
fig1, axes1 = plt.subplots(1, 3, figsize=(18, 5))
fig1.suptitle("Qwen2.5-3B-Instruct — Accuracy by Strategy & Dataset",
              fontsize=14, fontweight="bold")

for ci, ds in enumerate(DATASETS_Q2B):
    ax = axes1[ci]
    vals = [get_metric_q2b(s, ds, "exact") for s in STRATEGIES_Q2B]
    bars = ax.bar(x, vals, W, color=PALETTE_Q2B,
                  edgecolor="white", linewidth=0.8, zorder=3)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
                f"{v:.1f}%", ha="center", va="bottom",
                fontsize=9, fontweight="bold")
    ax.set_title(ds, fontsize=11, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([STRATEGY_LABELS_Q2B[s] for s in STRATEGIES_Q2B], fontsize=9)
    ax.set_ylim(0, 115)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_ylabel("Accuracy (%)", fontsize=9)
    ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

# Hallucination rate bar chart
fig2, axes2 = plt.subplots(1, 3, figsize=(18, 5))
fig2.suptitle("Qwen2.5-3B-Instruct — Hallucination Rate by Strategy & Dataset",
              fontsize=14, fontweight="bold")

for ci, ds in enumerate(DATASETS_Q2B):
    ax = axes2[ci]
    vals = [get_metric_q2b(s, ds, "hallucination_rate") for s in STRATEGIES_Q2B]
    bars = ax.bar(x, vals, W, color=PALETTE_Q2B,
                  edgecolor="white", linewidth=0.8, zorder=3)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
                f"{v:.1f}%", ha="center", va="bottom",
                fontsize=9, fontweight="bold")
    ax.set_title(ds, fontsize=11, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([STRATEGY_LABELS_Q2B[s] for s in STRATEGIES_Q2B], fontsize=9)
    ax.set_ylim(0, 115)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_ylabel("Hallucination Rate (%)", fontsize=9)
    ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

# Summary DataFrame
rows = []
for strategy in STRATEGIES_Q2B:
    for ds in DATASETS_Q2B:
        m = qwen2b_results.get(strategy, {}).get(ds, {})
        rows.append({
            "Model":                  "Qwen2.5-3B-Instruct",
            "Strategy":               STRATEGY_LABELS_Q2B[strategy],
            "Dataset":                ds,
            "Accuracy (%)":           m.get("exact", 0),
            "Hallucination Rate (%)": m.get("hallucination_rate", 100),
            "F1 (%)":                 m.get("f1") if ds == "TriviaQA" else None,
        })

qwen2b_summary = pd.DataFrame(rows)
pd.set_option("display.float_format", "{:.2f}".format)
print(qwen2b_summary.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

DATASETS   = ["TruthfulQA", "FEVER", "TriviaQA"]
STRATEGIES = ["zero_shot", "few_shot", "chain_of_thought"]
STRATEGY_LABELS = {
    "zero_shot":        "Zero-Shot",
    "few_shot":         "Few-Shot",
    "chain_of_thought": "Chain-of-Thought",
}


MODEL_LABELS = ["Pythia-70m", "Pythia-160m", "Pythia-410m",
                "Pythia-1b", "Pythia-2.8b", "Qwen-1.5B", "Qwen-3B"]
PYTHIA_SIZES = ["70m", "160m", "410m", "1b", "2.8b"]
COLORS       = ["#4F8EF7", "#F76F4F", "#4FD18E",
                "#F7C94F", "#F97316", "#A855F7", "#EC4899"]


W  = 0.12
xs = np.arange(len(STRATEGIES))

def get_pythia_hall(size, strategy, dataset):
    if strategy == "zero_shot":
        return all_results.get(size, {}).get(dataset, {}).get("hallucination_rate")
    else:
        return phase2_results.get(size, {}).get(strategy, {}).get(dataset, {}).get("hallucination_rate")

# Hallucination Rate chart
fig, axes = plt.subplots(1, len(DATASETS), figsize=(18, 6))
fig.suptitle("Hallucination Rate — Pythia Family vs Qwen Models\n(lower = better)",
             fontsize=13, fontweight="bold")

for ci, ds in enumerate(DATASETS):
    ax = axes[ci]
    for mi, (label, color) in enumerate(zip(MODEL_LABELS, COLORS)):
        vals = []
        for strategy in STRATEGIES:
            if label == "Qwen-1.5B":
                v = qwen_results.get(strategy, {}).get(ds, {}).get("hallucination_rate", 0) or 0
            elif label == "Qwen-3B":
                v = qwen2b_results.get(strategy, {}).get(ds, {}).get("hallucination_rate", 0) or 0
            else:
                v = get_pythia_hall(PYTHIA_SIZES[mi], strategy, ds) or 0
            vals.append(v)

        offset = (mi - 3) * W
        bars = ax.bar(xs + offset, vals, W, label=label,
                      color=color, edgecolor="white", linewidth=0.7, zorder=3)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.8,
                    f"{v:.0f}%", ha="center", va="bottom",
                    fontsize=6.5, fontweight="bold")

    ax.set_title(ds, fontsize=11, fontweight="bold")
    ax.set_xticks(xs)
    ax.set_xticklabels([STRATEGY_LABELS[s] for s in STRATEGIES], fontsize=9)
    ax.set_ylim(0, 115)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_ylabel("Hallucination Rate (%)", fontsize=9)
    ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if ci == 0:
        ax.legend(fontsize=8, loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
# Accuracy chart
fig2, axes2 = plt.subplots(1, len(DATASETS), figsize=(18, 6))
fig2.suptitle("Accuracy — Pythia Family vs Qwen Models\n(higher = better)",
              fontsize=13, fontweight="bold")

for ci, ds in enumerate(DATASETS):
    ax = axes2[ci]
    for mi, (label, color) in enumerate(zip(MODEL_LABELS, COLORS)):
        vals = []
        for strategy in STRATEGIES:
            if label == "Qwen-1.5B":
                hall = qwen_results.get(strategy, {}).get(ds, {}).get("hallucination_rate", 100) or 100
                v = 100 - hall
            elif label == "Qwen-3B":
                hall = qwen2b_results.get(strategy, {}).get(ds, {}).get("hallucination_rate", 100) or 100
                v = 100 - hall
            else:
                hall = get_pythia_hall(PYTHIA_SIZES[mi], strategy, ds)
                v = (100 - hall) if hall is not None else 0
            vals.append(v)

        offset = (mi - 3) * W
        bars = ax.bar(xs + offset, vals, W, label=label,
                      color=color, edgecolor="white", linewidth=0.7, zorder=3)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.8,
                    f"{v:.0f}%", ha="center", va="bottom",
                    fontsize=6.5, fontweight="bold")

    ax.set_title(ds, fontsize=11, fontweight="bold")
    ax.set_xticks(xs)
    ax.set_xticklabels([STRATEGY_LABELS[s] for s in STRATEGIES], fontsize=9)
    ax.set_ylim(0, 115)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_ylabel("Accuracy (%)", fontsize=9)
    ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if ci == 0:
        ax.legend(fontsize=8, loc="upper left")

plt.tight_layout()
plt.show()

In [ ]:
rows = []
for strategy in STRATEGIES:
    for ds in DATASETS:
        m = qwen_results.get(strategy, {}).get(ds, {})
        rows.append({
            "Model":                  "Qwen2.5-1.5B-Instruct",
            "Strategy":               STRATEGY_LABELS[strategy],
            "Dataset":                ds,
            "Accuracy (%)":           m.get("exact", 0),
            "Hallucination Rate (%)": m.get("hallucination_rate", 100),
            "F1 (%)":                 m.get("f1") if ds == "TriviaQA" else None,
        })

qwen_summary = pd.DataFrame(rows)
pd.set_option("display.float_format", "{:.2f}".format)
print(qwen_summary.to_string(index=False))

In [ ]:
# Qwen 3B summary
rows_3b = []
for strategy in STRATEGIES:
    for ds in DATASETS:
        m = qwen2b_results.get(strategy, {}).get(ds, {})
        rows_3b.append({
            "Model":                  "Qwen2.5-3B-Instruct",
            "Strategy":               STRATEGY_LABELS[strategy],
            "Dataset":                ds,
            "Accuracy (%)":           m.get("exact", 0),
            "Hallucination Rate (%)": m.get("hallucination_rate", 100),
            "F1 (%)":                 m.get("f1") if ds == "TriviaQA" else None,
        })

qwen3b_summary = pd.DataFrame(rows_3b)
print(qwen3b_summary.to_string(index=False))

### Statistical Significance Testing — Qwen Models Only

In [ ]:
!pip install -q statsmodels
from statsmodels.stats.proportion import proportions_ztest
import numpy as np
import pandas as pd

def proportion_ztest(acc1_pct, acc2_pct, n=500):
    count = np.array([round(acc1_pct / 100 * n), round(acc2_pct / 100 * n)])
    nobs  = np.array([n, n])
    stat, p_val = proportions_ztest(count, nobs)
    return round(float(p_val), 4), round(float(stat), 4)

def get_qwen_metric(model_key, strategy, ds, metric):

    if model_key == "qwen_1.5b":
        return qwen_results.get(strategy, {}).get(ds, {}).get(metric, 0) or 0
    elif model_key == "qwen_3b":
        return qwen2b_results.get(strategy, {}).get(ds, {}).get(metric, 0) or 0

def get_qwen_metric_for_ds(model_key, strategy, ds):
    metric = "f1" if ds == "TriviaQA" else "exact"
    return get_qwen_metric(model_key, strategy, ds, metric)

QWEN_MODELS    = ["qwen_1.5b", "qwen_3b"]
QWEN_DISPLAY   = {"qwen_1.5b": "Qwen2.5-1.5B", "qwen_3b": "Qwen2.5-3B"}
DATASETS_Q     = ["TruthfulQA", "FEVER", "TriviaQA"]
STRATEGIES_Q   = ["zero_shot", "few_shot", "chain_of_thought"]
STRAT_DISPLAY  = {
    "zero_shot":         "Zero-Shot",
    "few_shot":          "Few-Shot",
    "chain_of_thought":  "CoT",
}
strategy_pairs = [
    ("zero_shot",  "few_shot",         "ZS vs FS"),
    ("zero_shot",  "chain_of_thought", "ZS vs CoT"),
    ("few_shot",   "chain_of_thought", "FS vs CoT"),
]

print("Qwen models:  ", [QWEN_DISPLAY[m] for m in QWEN_MODELS])
print("Datasets:     ", DATASETS_Q)
print("Strategies:   ", list(STRAT_DISPLAY.values()))


# 1. QWEN-1.5B vs QWEN-3B — Zero-Shot
print(f"\n{'═'*75}")
print("  1. Qwen-1.5B vs Qwen-3B — Zero-Shot (Two-Proportion Z-Test)")
print(f"{'═'*75}")
print(f"{'Comparison':<30} {'Dataset':<12} {'Metric':<10} {'Z-stat':>8} {'p-value':>10} {'Sig?':>8}")
print("-"*75)

for ds in DATASETS_Q:
    acc1   = get_qwen_metric_for_ds("qwen_1.5b", "zero_shot", ds)
    acc2   = get_qwen_metric_for_ds("qwen_3b",   "zero_shot", ds)
    p, z   = proportion_ztest(acc1, acc2)
    sig    = "✓ p<0.05" if p < 0.05 else "✗ n.s."
    metric = "F1" if ds == "TriviaQA" else "Accuracy"
    print(f"{'Qwen-1.5B vs Qwen-3B':<30} {ds:<12} {metric:<10} {z:>8.3f} {p:>10.4f} {sig:>8}")

# 2. QWEN-1.5B vs QWEN-3B — Across All Strategies
print(f"\n{'═'*75}")
print("  2. Qwen-1.5B vs Qwen-3B — All Strategies (Two-Proportion Z-Test)")
print(f"{'═'*75}")
print(f"{'Strategy':<22} {'Dataset':<12} {'Metric':<10} {'Z-stat':>8} {'p-value':>10} {'Sig?':>8}")
print("-"*75)

for strategy in STRATEGIES_Q:
    for ds in DATASETS_Q:
        acc1   = get_qwen_metric_for_ds("qwen_1.5b", strategy, ds)
        acc2   = get_qwen_metric_for_ds("qwen_3b",   strategy, ds)
        p, z   = proportion_ztest(acc1, acc2)
        sig    = "✓ p<0.05" if p < 0.05 else "✗ n.s."
        metric = "F1" if ds == "TriviaQA" else "Accuracy"
        print(f"{STRAT_DISPLAY[strategy]:<22} {ds:<12} {metric:<10} {z:>8.3f} {p:>10.4f} {sig:>8}")
    print()


# 3. STRATEGY COMPARISONS — per Qwen model
print(f"\n{'═'*75}")
print("  3. Prompting Strategy Comparisons — per Qwen Model")
print(f"{'═'*75}")
print(f"{'Model':<18} {'Comparison':<22} {'Dataset':<12} {'Metric':<10} {'p-value':>10} {'Sig?':>8}")
print("-"*75)

for model_key in QWEN_MODELS:
    for s1_key, s2_key, label in strategy_pairs:
        for ds in DATASETS_Q:
            acc1   = get_qwen_metric_for_ds(model_key, s1_key, ds)
            acc2   = get_qwen_metric_for_ds(model_key, s2_key, ds)
            p, _   = proportion_ztest(acc1, acc2)
            sig    = "✓ p<0.05" if p < 0.05 else "✗ n.s."
            metric = "F1" if ds == "TriviaQA" else "Accuracy"
            print(f"{QWEN_DISPLAY[model_key]:<18} {label:<22} {ds:<12} {metric:<10} {p:>10.4f} {sig:>8}")
    print()


# 4. FULL SUMMARY TABLE
sig_rows = []

# Section 1 & 2: 1.5B vs 3B across all strategies
for strategy in STRATEGIES_Q:
    for ds in DATASETS_Q:
        acc1   = get_qwen_metric_for_ds("qwen_1.5b", strategy, ds)
        acc2   = get_qwen_metric_for_ds("qwen_3b",   strategy, ds)
        p, z   = proportion_ztest(acc1, acc2)
        metric = "F1" if ds == "TriviaQA" else "Accuracy"
        sig_rows.append({
            "Section":     "Model Size",
            "Comparison":  f"Qwen-1.5B vs Qwen-3B | {STRAT_DISPLAY[strategy]}",
            "Dataset":     ds,
            "Metric":      metric,
            "p-value":     p,
            "Significant": "Yes ✓" if p < 0.05 else "No ✗"
        })

# Section 3: strategy comparisons per model
for model_key in QWEN_MODELS:
    for s1_key, s2_key, label in strategy_pairs:
        for ds in DATASETS_Q:
            acc1   = get_qwen_metric_for_ds(model_key, s1_key, ds)
            acc2   = get_qwen_metric_for_ds(model_key, s2_key, ds)
            p, _   = proportion_ztest(acc1, acc2)
            metric = "F1" if ds == "TriviaQA" else "Accuracy"
            sig_rows.append({
                "Section":     "Strategy",
                "Comparison":  f"{QWEN_DISPLAY[model_key]} | {label}",
                "Dataset":     ds,
                "Metric":      metric,
                "p-value":     p,
                "Significant": "Yes ✓" if p < 0.05 else "No ✗"
            })

sig_df_qwen = pd.DataFrame(sig_rows)
pd.set_option("display.max_rows",    150)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width",       140)
pd.set_option("display.float_format", "{:.4f}".format)

print(f"\n{'═'*75}")
print("  4. Full Summary Table — All Qwen Tests")
print(f"{'═'*75}")
print(sig_df_qwen.to_string(index=False))

sig_only_qwen = sig_df_qwen[sig_df_qwen["Significant"] == "Yes ✓"]
print(f"\n── Significant only (p < 0.05) ──")
print(sig_only_qwen[["Section","Comparison","Dataset","Metric","p-value"]].to_string(index=False))

print(f"\n── Counts ──")
print(f"Total tests run:      {len(sig_df_qwen)}")
print(f"Significant (p<0.05): {len(sig_only_qwen)}")
print(f"Non-significant:      {len(sig_df_qwen) - len(sig_only_qwen)}")

print(f"\n── Breakdown by Section ──")
for section in sig_df_qwen["Section"].unique():
    sub   = sig_df_qwen[sig_df_qwen["Section"] == section]
    n_sig = (sub["Significant"] == "Yes ✓").sum()
    print(f"  {section:<20}: {n_sig}/{len(sub)} significant")

print(f"\n── Breakdown by Dataset ──")
for ds in DATASETS_Q:
    sub   = sig_df_qwen[sig_df_qwen["Dataset"] == ds]
    n_sig = (sub["Significant"] == "Yes ✓").sum()
    print(f"  {ds:<12}: {n_sig}/{len(sub)} significant")

print(f"\n── Breakdown by Model ──")
for model_key in QWEN_MODELS:
    sub   = sig_df_qwen[sig_df_qwen["Comparison"].str.contains(QWEN_DISPLAY[model_key])]
    n_sig = (sub["Significant"] == "Yes ✓").sum()
    print(f"  {QWEN_DISPLAY[model_key]:<18}: appears in {len(sub)} tests, {n_sig} significant")

### Final Summary — Does Model Size Affect Hallucination Rate? (All Strategies)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
from matplotlib.lines import Line2D

DATASETS_FINAL   = ["TruthfulQA", "FEVER", "TriviaQA"]
PYTHIA_KEYS      = [s for s in ["70m","160m","410m","1b","2.8b"] if s in all_results]
PYTHIA_PARAMS    = {"70m": 70, "160m": 160, "410m": 410, "1b": 1000, "2.8b": 2800}
STRATEGIES_FINAL = ["zero_shot", "few_shot", "chain_of_thought"]
STRAT_LABELS_F   = {"zero_shot": "Zero-Shot", "few_shot": "Few-Shot", "chain_of_thought": "CoT"}
STRAT_COLORS_F   = {"zero_shot": "#4F8EF7",   "few_shot": "#F76F4F",  "chain_of_thought": "#4FD18E"}
STRAT_MARKERS_F  = {"zero_shot": "o",          "few_shot": "s",        "chain_of_thought": "^"}


QWEN_POINTS = {
    "Qwen-1.5B": (3600, qwen_results),
    "Qwen-3B":   (4400, qwen2b_results),
}
ALL_XTICKS       = [70, 160, 410, 1000, 2800, 3600, 4400]
ALL_XTICKLABELS  = ["70M", "160M", "410M", "1B", "2.8B", "Qwen\n1.5B", "Qwen\n3B"]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle("Does Model Size Reduce Hallucination Rate?\nPythia base models (lines) vs Qwen Instruct (diamonds)",
             fontsize=13, fontweight="bold")

for ax, ds in zip(axes, DATASETS_FINAL):

    # Pythia lines
    for strategy in STRATEGIES_FINAL:
        px = [PYTHIA_PARAMS[s] for s in PYTHIA_KEYS]
        py = [
            all_results[s][ds]["hallucination_rate"] if strategy == "zero_shot"
            else phase2_results[s][strategy][ds]["hallucination_rate"]
            for s in PYTHIA_KEYS
        ]
        ax.plot(px, py,
                color=STRAT_COLORS_F[strategy],
                marker=STRAT_MARKERS_F[strategy],
                linewidth=2, markersize=7,
                label=STRAT_LABELS_F[strategy])

    ax.axvline(x=3000, color="gray", linestyle="--",
               linewidth=1.2, alpha=0.6)
    ax.text(3050, 108, "← Pythia  |  Qwen →",
            fontsize=7.5, color="gray", fontstyle="italic")

    VERT_OFFSETS = {"zero_shot": 6, "few_shot": 0, "chain_of_thought": -6}

    for q_label, (q_params, q_res) in QWEN_POINTS.items():
        for strategy in STRATEGIES_FINAL:
            val = q_res.get(strategy, {}).get(ds, {}).get("hallucination_rate", None)
            if val is not None:
                plot_y = val + VERT_OFFSETS[strategy]
                ax.scatter(q_params, plot_y,
                           color=STRAT_COLORS_F[strategy],
                           marker="D", s=100, zorder=5,
                           edgecolors="white", linewidth=0.8)
                ax.annotate(f"{val:.0f}%",
                            xy=(q_params, plot_y),
                            xytext=(10, 0),
                            textcoords="offset points",
                            ha="left", va="center",
                            fontsize=7.5, fontweight="bold",
                            color=STRAT_COLORS_F[strategy])

    qwen_handle = Line2D([0],[0], marker="D", color="gray",
                         linewidth=0, markersize=8,
                         label="Qwen Instruct (◆)")
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles + [qwen_handle],
              labels  + ["Qwen Instruct (◆)"],
              fontsize=7.5, loc="upper right")

    ax.set_title(ds, fontsize=12, fontweight="bold")
    ax.set_xlabel("Model", fontsize=9)
    ax.set_ylabel("Hallucination Rate (%)", fontsize=9)
    ax.set_ylim(0, 118)
    ax.set_xlim(-100, 5000)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_xticks(ALL_XTICKS)
    ax.set_xticklabels(ALL_XTICKLABELS, fontsize=7.5)
    ax.grid(linestyle="--", alpha=0.4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

# Text Summary 1 — Pythia scaling: 70M → 2.8B
print("\n── Pythia: Does scaling reduce hallucination? (70M → 2.8B) ──\n")
print(f"{'Dataset':<12} {'Strategy':<18} {'70M':>6} {'2.8B':>6} {'Δ':>7}  Verdict")
print("-"*65)
for ds in DATASETS_FINAL:
    for strategy in STRATEGIES_FINAL:
        if "70m" in all_results and "2.8b" in all_results:
            if strategy == "zero_shot":
                small = all_results["70m"][ds]["hallucination_rate"]
                large = all_results["2.8b"][ds]["hallucination_rate"]
            else:
                small = phase2_results["70m"][strategy][ds]["hallucination_rate"]
                large = phase2_results["2.8b"][strategy][ds]["hallucination_rate"]
            delta   = large - small
            verdict = "✓ Improves" if delta < -2 else ("✗ Worsens" if delta > 2 else "→ Flat")
            print(f"{ds:<12} {STRAT_LABELS_F[strategy]:<18} "
                  f"{small:>5.1f}% {large:>5.1f}% {delta:>+6.1f}%  {verdict}")
    print()

# Text Summary 2 — Qwen Instruct vs Pythia-2.8B
print("\n── Qwen Instruct vs Pythia-2.8B ──\n")
print(f"{'Dataset':<12} {'Model':<12} {'Strategy':<18} {'Halluc.':>8}  {'vs Pythia-2.8B':>16}")
print("-"*72)
for ds in DATASETS_FINAL:
    for q_label, (_, q_res) in QWEN_POINTS.items():
        for strategy in STRATEGIES_FINAL:
            val = q_res.get(strategy, {}).get(ds, {}).get("hallucination_rate", None)
            if val is not None:
                p28 = (all_results["2.8b"][ds]["hallucination_rate"]
                       if strategy == "zero_shot"
                       else phase2_results["2.8b"][strategy][ds]["hallucination_rate"])
                diff   = val - p28
                better = "✓ Better" if diff < -2 else ("✗ Worse" if diff > 2 else "→ Similar")
                print(f"{ds:<12} {q_label:<12} {STRAT_LABELS_F[strategy]:<18} "
                      f"{val:>7.1f}%  {better} ({diff:>+.1f}%)")
    print()